# DataInsight | Tabular: Dự đoán hủy đặt phòng khách sạn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/tabular.ipynb)

| | |
|---|---|
| **Môn học** | Nền tảng lập trình cho phân tích và trực quan dữ liệu (CO5177), Học kỳ 261, 2026–2027 |
| **Giảng viên** | Lê Thành Sách, Khoa Khoa học và Kỹ thuật Máy tính, ĐH Bách Khoa – ĐHQG-HCM |
| **Nhóm** | DataInsight: Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Người thực hiện** | Trần Lê Hoàng Long (2570443) |
| **Dataset** | [Hotel Booking Demand](https://github.com/rfordatascience/tidytuesday/tree/master/data/2020/2020-02-11) (António, Almeida & Nunes, 2019, *Data in Brief* 22:41–49), bản TidyTuesday 2020-02-11, CC BY 4.0 |
| **Bài toán** | Phân loại nhị phân cột `is_canceled` (1 là booking bị hủy) |

**Mô tả bài toán.** Khi một booking bị hủy sát ngày, khách sạn thường không kịp bán lại phòng. Nếu biết trước booking nào dễ bị hủy, khách sạn có thể gọi xác nhận, yêu cầu đặt cọc hoặc nhận thêm booking bù vào. Bài này dùng 119.390 booking của một resort và một khách sạn trong thành phố ở Bồ Đào Nha (ngày đến từ 07/2015 đến 08/2017) để dự đoán một booking có bị hủy hay không. Mô hình chỉ được dùng những thông tin có tại lúc khách đặt phòng như thời gian đặt trước, số đêm, số khách, kênh bán, loại đặt cọc, giá phòng và lịch sử của khách. Những cột chỉ được ghi sau đó, ví dụ `reservation_status`, bị loại vì chúng cho biết luôn kết quả. Do số booking bị hủy ít hơn số booking không hủy, các mô hình được so sánh chủ yếu bằng F1 và PR-AUC của lớp hủy thay vì accuracy.

**Mục lục**
1. [Cài đặt](#s1)
2. [Tải dữ liệu](#s2)
3. [Mô tả tập dữ liệu](#s3)
4. [Kiểm tra chất lượng dữ liệu](#s4)
5. [Phân tích khám phá (EDA)](#s5)
6. [Chuẩn bị dữ liệu](#s6)
7. [Mô hình hóa](#s7)
8. [Đánh giá trên tập test](#s8)
9. [Dự đoán trên dữ liệu mới](#s9)
10. [Tổng kết](#s10)

<a id="s1"></a>
## 1. Cài đặt

### 1.1 Cài thư viện còn thiếu

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.2",
    "plotly": "plotly>=5.15",
    "imblearn": "imbalanced-learn>=0.12",
    "gdown": "gdown>=4.6",
    "kagglehub": "kagglehub",
}

# find_spec trả về None khi module chưa được cài; chưa import gì nên không cần restart.
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Đang cài:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Đã có đủ thư viện cần thiết.")

Đã có đủ thư viện cần thiết.


### 1.2 Import thư viện

In [2]:
import itertools
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np                         # mảng, tính toán vector hóa
import pandas as pd                        # bảng: đọc, làm sạch, gom nhóm, tách X / y
import plotly.express as px                # biểu đồ Plotly nhanh từ DataFrame
import plotly.graph_objects as go          # biểu đồ Plotly tùy chỉnh chi tiết
import plotly.io as pio                    # template Plotly dùng chung
from plotly.subplots import make_subplots  # nhiều panel trong một hình

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone  # clone: bản sao chưa fit; 2 lớp kia để viết transformer riêng
from sklearn.compose import ColumnTransformer                   # tiền xử lý khác nhau cho từng nhóm cột
from sklearn.dummy import DummyClassifier                       # baseline bỏ qua đặc trưng
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer            # điền missing bằng thống kê của train
from sklearn.inspection import permutation_importance           # độ quan trọng đặc trưng, dùng được cho mọi mô hình
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (FixedThresholdClassifier, RandomizedSearchCV, StratifiedKFold,
                                     TunedThresholdClassifierCV,
                                     train_test_split)
from sklearn.pipeline import Pipeline                           # nối tiền xử lý + mô hình
from sklearn.preprocessing import (FunctionTransformer, LabelEncoder, OneHotEncoder, OrdinalEncoder,
                                   RobustScaler, StandardScaler)
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE                         # sinh mẫu lớp thiểu số (mục 7.3)
from imblearn.pipeline import Pipeline as ImbPipeline           # pipeline cho phép bước resample, chỉ chạy lúc fit

warnings.filterwarnings("ignore", category=FutureWarning)  # chỉ ẩn cảnh báo deprecation của thư viện; giữ cảnh báo thật
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1


### 1.3 Cấu hình

Các thông số phụ thuộc vào dataset (nguồn dữ liệu, cột nhãn, các cột bị loại, quy tắc kiểm tra giá trị) được gom vào cell này để dễ theo dõi và chỉnh sửa.

In [3]:
# ---- Tái lập kết quả ---------------------------------------------------------------------------
SEED = 42                     # một seed cho mọi bước ngẫu nhiên (split, mô hình, CV) -> Run all ra cùng số liệu
np.random.seed(SEED)

# ---- Nguồn dữ liệu (mục 2) ---------------------------------------------------------------------
DATA_DIR = Path("data")       # cache local, đã gitignore
DATA_SOURCE = {
    "kind": "url",            # file CSV công khai trên GitHub: không cần đăng nhập hay API key
    "location": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-02-11/hotels.csv",
    "filename": "hotels.csv", # giống hệt file của Kaggle "jessemostipak/hotel-booking-demand"
}
READ_KWARGS = {}              # tham số thêm cho pandas, vd {"sep": ";", "encoding": "latin-1"}

# ---- Định nghĩa bài toán -----------------------------------------------------------------------
TARGET = "is_canceled"        # 1 = booking bị hủy (kể cả no-show), 0 = khách đã check-out
ID_COLS = []                  # cột định danh (customer_id, ...): giữ để tra cứu, không dùng làm đặc trưng
DROP_COLS = [                 # cột loại bỏ có chủ đích; giải thích ở mục 6.1
    "reservation_status",        # leakage: Canceled / No-Show / Check-Out chính là nhãn
    "reservation_status_date",   # leakage: ngày cập nhật trạng thái, chỉ biết sau khi có kết quả
    "assigned_room_type",        # chỉ biết lúc check-in
    "booking_changes",           # đếm số lần sửa booking đến lúc check-in hoặc hủy
    "arrival_date_year",         # chỉ có 2015-2017: mô hình không học được quy luật cho năm tương lai
]
FORCE_CATEGORICAL = ["agent", "company"]  # mã ID ẩn danh của đại lý / công ty, không phải đại lượng
NA_TOKENS = ["?", "NA", "N/A", "n/a", "null", "NULL", ""]  # các chuỗi mang nghĩa "thiếu" trong file

# ---- Quy tắc miền (mục 4.4 và 6.1) -------------------------------------------------------------
VALID_RANGES = {              # theo kiến thức miền, không theo thống kê; None = không giới hạn
    "adr": (0, 1000),            # không thể âm; 5.400 gấp hơn 10 lần giá cao thứ hai (510) -> lỗi nhập liệu (mục 4.4)
    "adults": (0, None),
    "lead_time": (0, None),
}
SPELLING_MAPS = {             # key đã strip + viết thường
    "meal": {"undefined": "SC"}, # theo bài báo gốc: "Undefined" và "SC" đều là không có gói ăn
}
TO_NUMERIC = []               # không có: mọi cột số đều đã được đọc thành số
ROW_RULES = {                 # quy tắc trên nhiều cột (mục 4.4): hàm trả về True cho dòng hợp lệ
    "có ít nhất 1 khách": lambda d: d["adults"] + d["children"].fillna(0) + d["babies"] > 0,
}

# ---- Chia dữ liệu và tiền xử lý ----------------------------------------------------------------
TEST_SIZE, VAL_SIZE = 0.15, 0.15   # 70% train / 15% validation / 15% test
MIN_CATEGORY_FREQ = 0.01           # category hiếm hơn 1% của train được gộp vào một cột "infrequent"
ORDINAL_COLS = {                   # category có thứ tự tự nhiên -> OrdinalEncoder theo đúng thứ tự này
    "arrival_date_month": ["January", "February", "March", "April", "May", "June", "July",
                           "August", "September", "October", "November", "December"],
}
OUTLIER_COLS = ["lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights",  # cột liên tục đuôi dài
                "total_nights", "days_in_waiting_list"]                                # (mục 4.5, 6.5)
CLIP_QUANTILES = (0.01, 0.99)      # ngưỡng cắt outlier theo phân vị, học trên train (mục 6.5)

# Dừng Run all sớm với thông báo rõ ràng thay vì lỗi ở 20 cell sau.
for _name, _value in {"DATA_SOURCE": DATA_SOURCE, "TARGET": TARGET}.items():
    assert "TODO" not in str(_value), f"Hãy điền {_name} ở mục 1.3 trước khi chạy notebook."

### 1.4 Định dạng biểu đồ

Các biểu đồ được vẽ bằng Plotly, dùng chung bảng màu và font với notebook Time series của nhóm. Hàm `show_fig` đánh số hình theo thứ tự để phần nhận xét có thể nhắc lại số hình.

In [4]:
# Màu theo vai trò, thứ tự cố định: chuỗi phân loại dùng BLUE, ORANGE, AQUA, YELLOW, rồi RED/MUTED.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
QUALITATIVE = [BLUE, ORANGE, AQUA, YELLOW, RED, MUTED]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # số đếm, cường độ
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]                                          # tương quan (-1..1)

# Một template Plotly có tên = một chỗ duy nhất để đổi giao diện mọi biểu đồ.
pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=QUALITATIVE,
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=70, b=50),
))
pio.templates.default = "plotly_white+datainsight"

_figure_number = itertools.count(1)  # bộ đếm toàn cục: Run all đánh số hình 1, 2, 3, ... theo thứ tự


def show_fig(fig, title, height=420, **layout):
    """Đánh số hình, gắn tiêu đề dạng kết luận rồi hiển thị."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE, name=None):
    """Histogram tính sẵn bằng NumPy: notebook chỉ lưu ~40 cột thay vì mọi điểm dữ liệu."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


# Tên metric và tên cách xử lý trên hình viết tiếng Anh (bảng kết quả vẫn giữ tên tiếng Việt)
LABEL_EN = {"precision (hủy)": "precision (cancelled)", "recall (hủy)": "recall (cancelled)",
            "F1 (hủy)": "F1 (cancelled)", "(a) bỏ dòng thiếu": "(a) drop incomplete rows",
            "(b) bỏ cột + median/mode": "(b) drop column + median/mode", "(c) missing = category": "(c) missing as category",
            "(d) KNN + category": "(d) KNN imputer + category", "giữ nguyên": "keep as is",
            "cắt phân vị 1%–99%": "clip at 1st/99th percentile"}


def grid_position(i, ncols):
    """(hàng, cột) của panel thứ i trong lưới make_subplots, điền từ trái sang phải."""
    return i // ncols + 1, i % ncols + 1

<a id="s2"></a>
## 2. Tải dữ liệu

### 2.1 Tải về

Dữ liệu được tải trực tiếp từ kho TidyTuesday trên GitHub (file `hotels.csv`, giống với bản trên Kaggle) và lưu tạm vào thư mục `data/`. Lần chạy sau sẽ đọc lại file đã tải. Hàm `download_dataset` cũng hỗ trợ link Google Drive, Kaggle và file `.zip`.

In [5]:
def download_dataset(source, data_dir=DATA_DIR):
    """Tải một nguồn dữ liệu (bỏ qua nếu đã có cache) và trả về đường dẫn của `source["filename"]`."""
    data_dir.mkdir(parents=True, exist_ok=True)
    kind, location, filename = source["kind"], source["location"], source["filename"]

    if kind == "kaggle":
        import kagglehub
        root = Path(kagglehub.dataset_download(location))  # kagglehub có cache riêng
    else:
        root = data_dir
        archive = source.get("archive")
        target = data_dir / (archive or filename)
        if not target.exists():  # cache: không bao giờ tải hai lần
            if kind == "url":
                urllib.request.urlretrieve(location, target)
            elif kind == "gdrive":
                import gdown
                gdown.download(id=location, output=str(target), quiet=False)
            else:
                raise ValueError(f"Loại nguồn không hợp lệ {kind!r}: dùng 'url', 'gdrive' hoặc 'kaggle'.")
        if archive:
            with zipfile.ZipFile(target) as zf:
                zf.extractall(data_dir)

    matches = sorted(root.rglob(filename))  # rglob: file có thể nằm trong thư mục con của file nén
    if not matches:
        found = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()][:20]
        raise FileNotFoundError(f"Không tìm thấy {filename!r} trong {root}. Các file có: {found}")
    print(f"File dữ liệu: {matches[0]} ({matches[0].stat().st_size / 1e6:,.1f} MB)")
    return matches[0]


def read_table(path, na_values=None, **kwargs):
    """Đọc file dữ liệu bằng hàm đọc pandas khớp với phần mở rộng."""
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt", ".data"}:
        return pd.read_csv(path, na_values=na_values, **kwargs)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", na_values=na_values, **kwargs)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, na_values=na_values, **kwargs)
    if suffix == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl", **kwargs)
    raise ValueError(f"Chưa có hàm đọc cho {suffix!r}; thêm vào read_table.")

### 2.2 Đọc bảng dữ liệu

In [6]:
data_path = download_dataset(DATA_SOURCE)
raw = read_table(data_path, na_values=NA_TOKENS, **READ_KWARGS)
raw.columns = raw.columns.str.strip()  # tên cột thừa khoảng trắng sẽ làm lỗi khi truy cập cột về sau
assert TARGET in raw.columns, f"Không tìm thấy TARGET {TARGET!r}. Các cột: {list(raw.columns)}"

print(f"Kích thước: {raw.shape[0]:,} dòng x {raw.shape[1]} cột")
raw.head()

File dữ liệu: data/hotels.csv (16.9 MB)
Kích thước: 119,390 dòng x 32 cột


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.000,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.000,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.000,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.000,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.000,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.000,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.000,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.000,NaN,0,Transient,75.000,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.000,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.000,NaN,0,Transient,98.000,0,1,Check-Out,2015-07-03


### 2.3 Kiểm tra yêu cầu của đề bài

Đề bài yêu cầu dữ liệu dạng bảng có ít nhất 2.000 mẫu, 10 cột, có missing values và có cả cột categorical lẫn numerical.

In [7]:
_features = [c for c in raw.columns if c not in [TARGET, *ID_COLS]]
_n_numeric = len(raw[_features].select_dtypes("number").columns)
requirements = pd.DataFrame([
    ("Ít nhất 2.000 mẫu", f"{len(raw):,} dòng", len(raw) >= 2000),
    ("Ít nhất 10 cột", f"{raw.shape[1]} cột", raw.shape[1] >= 10),
    ("Có missing values", f"{int(raw.isna().sum().sum()):,} ô thiếu", bool(raw.isna().any().any())),
    ("Có cột categorical", f"{len(_features) - _n_numeric} cột không phải số", len(_features) > _n_numeric),
    ("Có cột numerical", f"{_n_numeric} cột số", _n_numeric > 0),
], columns=["Yêu cầu (đề bài §4.2)", "Dataset này", "Đạt"])
requirements

,Yêu cầu (đề bài §4.2),Dataset này,Đạt
0,Ít nhất 2.000 mẫu,"119,390 dòng",True
1,Ít nhất 10 cột,32 cột,True
2,Có missing values,"129,425 ô thiếu",True
3,Có cột categorical,12 cột không phải số,True
4,Có cột numerical,19 cột số,True


**Nhận xét.** Dataset đáp ứng cả năm yêu cầu với 119.390 dòng và 32 cột. Có 129.425 ô bị thiếu, tập trung ở `company` (94,3%), `agent` (13,7%), `country` (0,4%) và 4 ô của `children`; trong file các ô này được ghi là `NULL` hoặc `NA`. Ngoài cột nhãn có 12 cột chữ và 19 cột số. Riêng `agent` và `company` tuy lưu dưới dạng số nhưng thực chất là mã ID của đại lý và công ty, nên ở mục 3 được xếp vào nhóm categorical.

<a id="s3"></a>
## 3. Mô tả tập dữ liệu

Dữ liệu được trích từ hệ thống quản lý khách sạn (PMS) của hai khách sạn ở Bồ Đào Nha: một resort ở vùng Algarve (H1, 40.060 booking) và một khách sạn trong thành phố Lisbon (H2, 79.330 booking). Bộ dữ liệu gồm tất cả booking có ngày đến từ 01/07/2015 đến 31/08/2017, kể cả booking đã hủy, và đã được xóa thông tin nhận dạng khách hàng. Bản sử dụng ở đây là bản TidyTuesday (2020-02-11), trong đó hai bảng H1, H2 đã được gộp lại và thêm cột `hotel`. Nguồn gốc: N. António, A. de Almeida, L. Nunes (2019), *Hotel booking demand datasets*, Data in Brief 22:41–49.

Mỗi dòng là một booking. Vì mục tiêu là dự đoán ngay tại lúc khách đặt phòng, cột "Có lúc đặt?" trong bảng dưới cho biết thông tin đó đã có tại thời điểm đó hay chỉ được ghi sau này. Những cột ghi sau này bị loại (`DROP_COLS`).

| Nhóm | Cột | Kiểu | Ý nghĩa | Có lúc đặt? | Sử dụng |
|---|---|---|---|---|---|
| Nhãn | `is_canceled` | 0/1 | Booking bị hủy (1) hay không (0) | | target |
| Khách sạn | `hotel` | categorical | Resort Hotel (H1) hoặc City Hotel (H2) | có | đặc trưng |
| Thời gian | `lead_time` | numeric | Số ngày từ lúc đặt đến ngày đến | có | đặc trưng |
| | `arrival_date_year` | numeric | Năm đến (2015–2017) | có | loại, vì không áp dụng được cho năm sau |
| | `arrival_date_month` | categorical | Tháng đến | có | đặc trưng |
| | `arrival_date_week_number`, `arrival_date_day_of_month` | numeric | Tuần trong năm, ngày trong tháng | có | đặc trưng |
| Lưu trú | `stays_in_weekend_nights`, `stays_in_week_nights` | numeric | Số đêm cuối tuần và số đêm trong tuần | có | đặc trưng |
| Khách | `adults`, `children`, `babies` | numeric | Số người lớn, trẻ em, em bé | có | đặc trưng |
| | `country` | categorical | Quốc gia của khách (mã ISO 3 chữ) | có | đặc trưng |
| | `is_repeated_guest` | 0/1 | Khách đã từng ở khách sạn | có | đặc trưng |
| | `previous_cancellations`, `previous_bookings_not_canceled` | numeric | Số booking trước đó đã hủy và không hủy | có | đặc trưng |
| | `customer_type` | categorical | Contract, Group, Transient, Transient-Party | có | đặc trưng |
| Kênh bán | `market_segment`, `distribution_channel` | categorical | Phân khúc thị trường và kênh phân phối (TA là đại lý du lịch, TO là hãng lữ hành) | có | đặc trưng |
| | `agent`, `company` | mã ID | Đại lý hoặc công ty đặt phòng; để trống nếu không có | có | đặc trưng (categorical) |
| Phòng, dịch vụ | `reserved_room_type` | categorical | Mã loại phòng đã đặt | có | đặc trưng |
| | `meal` | categorical | Gói ăn: SC/Undefined (không có), BB, HB, FB | có | đặc trưng |
| | `required_car_parking_spaces`, `total_of_special_requests` | numeric | Số chỗ đỗ xe và số yêu cầu đặc biệt | có | đặc trưng |
| Giá, đặt cọc | `adr` | numeric | Giá trung bình mỗi đêm (EUR) | có | đặc trưng |
| | `deposit_type` | categorical | No Deposit, Non Refund, Refundable | có | đặc trưng |
| | `days_in_waiting_list` | numeric | Số ngày nằm trong danh sách chờ trước khi được xác nhận | có | đặc trưng |
| Sau khi đặt | `assigned_room_type` | categorical | Loại phòng thực nhận lúc check-in | không | loại |
| | `booking_changes` | numeric | Số lần sửa booking cho đến lúc check-in hoặc hủy | không | loại |
| | `reservation_status`, `reservation_status_date` | categorical, ngày | Trạng thái cuối (Canceled, Check-Out, No-Show) và ngày cập nhật | không | loại, vì chính là kết quả |

In [8]:
raw.info()  # kiểu dữ liệu từng cột: object/str = chữ, int64/float64 = số; cột số có NaN sẽ thành float64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

### 3.1 Phân loại cột

Các cột được chia thành numeric và categorical dựa trên kiểu dữ liệu, riêng `agent` và `company` được xếp vào categorical (`FORCE_CATEGORICAL`). Hai danh sách này được dùng cho các biểu đồ ở mục 4–5 và cho bước tiền xử lý ở mục 6.

In [9]:
def column_roles(df):
    """Chia các cột đặc trưng thành danh sách numeric và categorical, theo cấu hình."""
    features = [c for c in df.columns if c not in {TARGET, *ID_COLS, *DROP_COLS}]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(df[c]) and c not in FORCE_CATEGORICAL]
    categorical = [c for c in features if c not in numeric]
    return features, numeric, categorical


FEATURES, NUM_COLS, CAT_COLS = column_roles(raw)
print(f"{len(NUM_COLS)} cột numeric:", NUM_COLS)
print(f"{len(CAT_COLS)} cột categorical:", CAT_COLS)

15 cột numeric: ['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']
11 cột categorical: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'customer_type']


In [10]:
raw[NUM_COLS].describe().T  # count < số dòng => có missing; min/max lộ ra giá trị vô lý

,count,mean,std,min,25%,50%,75%,max
lead_time,"119,390.000",104.011,106.863,0.000,18.000,69.000,160.000,737.000
arrival_date_week_number,"119,390.000",27.165,13.605,1.000,16.000,28.000,38.000,53.000
arrival_date_day_of_month,"119,390.000",15.798,8.781,1.000,8.000,16.000,23.000,31.000
stays_in_weekend_nights,"119,390.000",0.928,0.999,0.000,0.000,1.000,2.000,19.000
stays_in_week_nights,"119,390.000",2.500,1.908,0.000,1.000,2.000,3.000,50.000
adults,"119,390.000",1.856,0.579,0.000,2.000,2.000,2.000,55.000
children,"119,386.000",0.104,0.399,0.000,0.000,0.000,0.000,10.000
babies,"119,390.000",0.008,0.097,0.000,0.000,0.000,0.000,10.000
is_repeated_guest,"119,390.000",0.032,0.176,0.000,0.000,0.000,0.000,1.000
previous_cancellations,"119,390.000",0.087,0.844,0.000,0.000,0.000,0.000,26.000


In [11]:
cat_summary = pd.DataFrame({
    "số giá trị khác nhau": raw[CAT_COLS].nunique(),
    "giá trị phổ biến nhất": raw[CAT_COLS].mode().iloc[0] if CAT_COLS else [],
    "tỷ lệ giá trị phổ biến nhất": [raw[c].value_counts(normalize=True).iloc[0] for c in CAT_COLS],
    "số ô thiếu": raw[CAT_COLS].isna().sum(),
})
cat_summary  # nhiều giá trị khác nhau = cardinality cao -> one-hot sẽ tạo nhiều cột (xem MIN_CATEGORY_FREQ)

,số giá trị khác nhau,giá trị phổ biến nhất,tỷ lệ giá trị phổ biến nhất,số ô thiếu
hotel,2,City Hotel,0.664,0
arrival_date_month,12,August,0.116,0
meal,5,BB,0.773,0
country,177,PRT,0.409,488
market_segment,8,Online TA,0.473,0
distribution_channel,5,TA/TO,0.820,0
reserved_room_type,10,A,0.720,0
deposit_type,3,No Deposit,0.876,0
agent,333,9.000,0.310,16340
company,352,40.000,0.136,112593


**Nhận xét.** Bảng gồm 119.390 dòng và 32 cột, trong đó có 16 cột số nguyên, 4 cột số thực và 12 cột chữ. Sau khi bỏ cột nhãn và 5 cột trong `DROP_COLS`, còn lại 26 đặc trưng gồm 15 cột numeric và 11 cột categorical. Cột `children` có kiểu số thực chỉ vì có 4 ô thiếu.

Một số cột categorical có rất nhiều giá trị: `country` có 177, `agent` 333 và `company` 352 giá trị. Nếu one-hot toàn bộ sẽ sinh ra hàng trăm cột, nên ở mục 6 những giá trị chiếm dưới 1% tập train được gộp chung (`MIN_CATEGORY_FREQ`). Ngược lại, có những cột gần như chỉ có một giá trị, ví dụ `distribution_channel` là TA/TO ở 82,0% số dòng, `deposit_type` là No Deposit ở 87,6%, `meal` là BB ở 77,3% và 66,4% booking thuộc City Hotel.

Bảng thống kê cũng cho thấy một số giá trị bất thường sẽ được kiểm tra ở mục 4.4–4.5: `adr` nằm trong khoảng từ −6,38 đến 5.400, có booking 55 người lớn, 10 trẻ em hoặc 10 em bé, có booking 50 đêm trong tuần và có booking chờ 391 ngày.

<a id="s4"></a>
## 4. Kiểm tra chất lượng dữ liệu

Mục này chỉ phát hiện vấn đề, còn việc xử lý được thực hiện ở mục 6. Các vấn đề được xem xét gồm missing values, dòng trùng, cách viết không thống nhất, giá trị vô lý, outlier và tỷ lệ giữa hai lớp.

### 4.1 Missing values

Ngoài các ô `NaN`, cell dưới cũng đếm các ô chứa chuỗi mang nghĩa thiếu (như `"?"` hay `" NA"`) mà pandas chưa nhận ra khi đọc file.

In [12]:
text_cols = raw.columns.difference(raw.select_dtypes("number").columns)
hidden = {c: int(raw[c].dropna().astype(str).str.strip().isin(NA_TOKENS).sum()) for c in text_cols}

missing = pd.DataFrame({"NaN": raw.isna().sum(), "chuỗi ẩn": pd.Series(hidden)}).fillna(0).astype(int)
missing["tổng"] = missing["NaN"] + missing["chuỗi ẩn"]
missing["% số dòng"] = missing["tổng"] / len(raw) * 100
missing = missing[missing["tổng"] > 0].sort_values("tổng", ascending=False)

if missing.empty:
    print("Không có missing values.")
else:
    fig = go.Figure(go.Bar(x=missing.index, y=missing["% số dòng"], marker_color=BLUE,
                           text=[f"{p:.1f}%" for p in missing["% số dòng"]], textposition="outside",
                           hovertemplate="%{x}: %{y:.2f}% of rows<extra></extra>"))
    show_fig(fig, f"{len(missing)} columns contain missing values (max {missing['% số dòng'].max():.1f}% of rows)",
             xaxis_title="Column", yaxis_title="% of rows missing")
missing

,NaN,chuỗi ẩn,tổng,% số dòng
company,112593,0,112593,94.307
agent,16340,0,16340,13.686
country,488,0,488,0.409
children,4,0,4,0.003


Tiếp theo, tỷ lệ hủy được so sánh giữa các dòng bị thiếu và không bị thiếu ở từng cột. Nếu hai tỷ lệ này khác nhau nhiều thì việc thiếu dữ liệu không xảy ra ngẫu nhiên và có thể dùng làm thông tin cho mô hình.

In [13]:
target_share = raw[TARGET].value_counts(normalize=True)
MINORITY = target_share.idxmin()  # lớp quan trọng nhất khi dữ liệu mất cân bằng

rows = []
for c in missing.index:
    is_na = raw[c].isna() | raw[c].astype(str).str.strip().isin(NA_TOKENS)
    if c == TARGET or is_na.all():
        continue
    rows.append({"cột": c,
                 f"% '{MINORITY}' khi thiếu": (raw.loc[is_na, TARGET] == MINORITY).mean() * 100,
                 f"% '{MINORITY}' khi có": (raw.loc[~is_na, TARGET] == MINORITY).mean() * 100})
pd.DataFrame(rows)

,cột,% '1' khi thiếu,% '1' khi có
0,company,38.220,17.522
1,agent,24.663,39.004
2,country,13.730,37.137
3,children,100.000,37.040


**Nhận xét.** Chỉ có 4 cột bị thiếu và không có chuỗi missing nào bị bỏ sót: `company` thiếu 112.593 ô (94,3%), `agent` 16.340 ô (13,7%), `country` 488 ô (0,4%) và `children` 4 ô.

Việc thiếu dữ liệu có liên quan rõ đến tỷ lệ hủy. Booking không có `company` bị hủy 38,2% trong khi booking có `company` chỉ hủy 17,5%. Với `agent` thì ngược lại, không có đại lý hủy 24,7%, có đại lý hủy 39,0%. Booking thiếu `country` chỉ hủy 13,7% so với 37,1% ở các booking còn lại. Cả 4 dòng thiếu `children` đều bị hủy nhưng số dòng quá ít để rút ra kết luận. Ô trống ở `agent` và `company` thực chất có nghĩa là booking không đặt qua đại lý hay công ty nào, nên không nên coi đó là lỗi ghi chép.

Từ đó, hướng xử lý ở mục 6 là giữ `agent` và `company` với một giá trị riêng cho trường hợp "không có" và thêm hai cột cờ `has_agent`, `has_company`. `country` và `children` thiếu rất ít nên được điền bằng giá trị phổ biến nhất hoặc median tính trên tập train. Cách bỏ các dòng bị thiếu không dùng được vì sẽ mất 99,8% dữ liệu do cột `company`. Mục 6.4 so sánh các cách này bằng kết quả mô hình.

### 4.2 Dòng trùng

Dòng trùng hoàn toàn có thể khiến cùng một mẫu nằm ở cả tập train và tập test, làm kết quả đánh giá cao hơn thực tế. Ngoài ra cần kiểm tra các nhóm có đặc trưng giống hệt nhau nhưng nhãn khác nhau, vì không mô hình nào phân biệt được các mẫu như vậy.

In [14]:
non_id = [c for c in raw.columns if c not in ID_COLS]
feature_like = [c for c in non_id if c != TARGET]

dup_table = pd.DataFrame({
    "kiểm tra": ["Dòng trùng hoàn toàn", "Trùng khi bỏ qua cột ID", "Giá trị ID lặp lại",
                 "Cùng đặc trưng, khác nhãn (số nhóm)"],
    "số lượng": [
        int(raw.duplicated().sum()),
        int(raw.duplicated(subset=non_id).sum()),
        int(raw.duplicated(subset=ID_COLS).sum()) if ID_COLS else 0,
        int((raw.groupby(feature_like, dropna=False)[TARGET].nunique() > 1).sum()),
    ],
})
dup_table

,kiểm tra,số lượng
0,Dòng trùng hoàn toàn,31994
1,Trùng khi bỏ qua cột ID,31994
2,Giá trị ID lặp lại,0
3,"Cùng đặc trưng, khác nhãn (số nhóm)",0


Dataset không có mã booking, nên khi một đoàn đặt nhiều phòng giống nhau thì các phòng đó trở thành nhiều dòng giống hệt nhau. Bảng dưới so sánh các bản sao thừa với những dòng còn lại để xem các dòng trùng thuộc loại booking nào.

In [15]:
extra_copy = raw.duplicated()                       # True cho bản sao thứ 2, 3, ... của một dòng
dup_profile = pd.DataFrame({
    "bản sao thừa": [extra_copy.sum(), raw.loc[extra_copy, TARGET].mean() * 100,
                     (raw.loc[extra_copy, "deposit_type"] == "Non Refund").mean() * 100,
                     (raw.loc[extra_copy, "market_segment"] == "Groups").mean() * 100],
    "các dòng còn lại": [(~extra_copy).sum(), raw.loc[~extra_copy, TARGET].mean() * 100,
                         (raw.loc[~extra_copy, "deposit_type"] == "Non Refund").mean() * 100,
                         (raw.loc[~extra_copy, "market_segment"] == "Groups").mean() * 100],
}, index=["số dòng", "% bị hủy", "% Non Refund", "% Groups"])
dup_profile.round(1)

,bản sao thừa,các dòng còn lại
số dòng,"31,994.000","87,396.000"
% bị hủy,63.100,27.500
% Non Refund,42.300,1.200
% Groups,46.500,5.700


**Nhận xét.** Có 31.994 bản sao thừa, chiếm 26,8% số dòng, và không có nhóm nào có đặc trưng giống nhau mà nhãn khác nhau. Tuy nhiên các bản sao rất khác phần còn lại: 63,1% trong số đó bị hủy (so với 27,5%), 42,3% là Non Refund (so với 1,2%) và 46,5% thuộc phân khúc Groups (so với 5,7%). Có tới 13.549 trong 14.587 booking Non Refund (92,9%) là bản sao, nên sau khi bỏ trùng chỉ còn khoảng 1.000 booking Non Refund (1.038 nếu xét đủ 32 cột, 992 nếu bỏ thêm các cột trong `DROP_COLS`, xem mục 5.7). Nhiều khả năng đây là các phòng của cùng một đoàn được nhập thành nhiều dòng.

Các dòng trùng hoàn toàn sẽ được bỏ trước khi chia tập (mục 6.1) để tránh cùng một booking xuất hiện ở cả train và test. Việc này làm tỷ lệ hủy giảm từ 37,0% xuống khoảng 27,7%, nên bài toán sau khi làm sạch mất cân bằng hơn so với dữ liệu gốc.

### 4.3 Cách viết không thống nhất và số lưu dạng chữ

Với pandas, `"HCM"`, `" HCM"` và `"hcm"` là ba giá trị khác nhau. Bảng dưới so sánh số giá trị khác nhau trước và sau khi xóa khoảng trắng và chuyển về chữ thường, đồng thời kiểm tra các cột chữ nhưng chứa toàn số.

In [16]:
rows, variant_examples = [], {}
for c in CAT_COLS:
    s = raw[c].dropna().astype(str)
    norm = s.str.strip().str.lower()
    rows.append({"cột": c,
                 "số giá trị (gốc)": s.nunique(),
                 "số giá trị (strip + lower)": norm.nunique(),
                 "giá trị có khoảng trắng ở đầu/cuối": int((s != s.str.strip()).sum()),
                 "tỷ lệ đọc được thành số": pd.to_numeric(s.str.replace(",", ""), errors="coerce").notna().mean()})
    groups = s.groupby(norm).unique()
    variants = groups[groups.map(len) > 1]
    if len(variants):
        variant_examples[c] = variants.head(5).map(list).to_dict()

display(pd.DataFrame(rows))
for c, examples in variant_examples.items():
    print(f"{c}: {examples}")

,cột,số giá trị (gốc),số giá trị (strip + lower),giá trị có khoảng trắng ở đầu/cuối,tỷ lệ đọc được thành số
0,hotel,2,2,0,0.000
1,arrival_date_month,12,12,0,0.000
2,meal,5,5,0,0.000
3,country,177,177,0,0.000
4,market_segment,8,8,0,0.000
5,distribution_channel,5,5,0,0.000
6,reserved_room_type,10,10,0,0.000
7,deposit_type,3,3,0,0.000
8,agent,333,333,0,1.000
9,company,352,352,0,1.000


**Nhận xét.** Số giá trị khác nhau trước và sau khi chuẩn hóa bằng nhau ở cả 11 cột, nghĩa là dữ liệu không có lỗi khoảng trắng hay viết hoa, viết thường lẫn lộn. `agent` và `company` đọc được hoàn toàn thành số, nhưng vì là mã ID nên vẫn giữ ở dạng categorical.

Tuy nhiên vẫn có trường hợp hai giá trị khác tên nhưng cùng nghĩa. Theo bài báo gốc, `meal = Undefined` (1.169 dòng) và `SC` đều có nghĩa là không đặt gói ăn, nên hai giá trị này được gộp lại qua `SPELLING_MAPS`. Giá trị `Undefined` ở `market_segment` (2 dòng) và `distribution_channel` (5 dòng) quá ít nên được để nguyên, khi mã hóa sẽ rơi vào nhóm giá trị hiếm.

### 4.4 Giá trị vô lý

Giá trị vô lý là những giá trị không thể xảy ra trong thực tế, ví dụ giá phòng âm. Khác với outlier, đây là lỗi dữ liệu. Các ngưỡng hợp lệ được đặt dựa trên ý nghĩa của từng cột trong `VALID_RANGES`.

In [17]:
rows = []
for c, (low, high) in VALID_RANGES.items():
    values = pd.to_numeric(raw[c], errors="coerce")
    too_low = int((values < low).sum()) if low is not None else 0
    too_high = int((values > high).sum()) if high is not None else 0
    rows.append({"cột": c, "quy tắc": f"[{low}, {high}]", "dưới ngưỡng": too_low, "trên ngưỡng": too_high,
                 "min quan sát": values.min(), "max quan sát": values.max()})
pd.DataFrame(rows) if rows else print("VALID_RANGES đang trống: thêm quy tắc miền ở mục 1.3.")

,cột,quy tắc,dưới ngưỡng,trên ngưỡng,min quan sát,max quan sát
0,adr,"[0, 1000]",1,1,-6.380,"5,400.000"
1,adults,"[0, None]",0,0,0.000,55.000
2,lead_time,"[0, None]",0,0,0.000,737.000


Một số lỗi chỉ phát hiện được khi xét nhiều cột cùng lúc, ví dụ booking không có khách nào hoặc không có đêm nào. Cell dưới kiểm tra các trường hợp này.

In [18]:
total_guests = raw["adults"] + raw["children"].fillna(0) + raw["babies"]
total_nights = raw["stays_in_weekend_nights"] + raw["stays_in_week_nights"]
checks = {
    "0 khách (adults + children + babies = 0)": total_guests == 0,
    "0 đêm (cuối tuần + trong tuần = 0)": total_nights == 0,
    "adr = 0": raw["adr"] == 0,
}
pd.DataFrame({
    "số dòng": {k: int(m.sum()) for k, m in checks.items()},
    "% bị hủy": {k: raw.loc[m, TARGET].mean() * 100 for k, m in checks.items()},
    "% có adr = 0": {k: (raw.loc[m, "adr"] == 0).mean() * 100 for k, m in checks.items()},
}).round(1)

,số dòng,% bị hủy,% có adr = 0
0 khách (adults + children + babies = 0),180,13.900,82.800
0 đêm (cuối tuần + trong tuần = 0),715,4.900,100.000
adr = 0,1959,10.900,100.000


**Nhận xét.** Có 1 booking có `adr` âm (−6,38) và 1 booking có `adr` bằng 5.400 (City Hotel, 2 người lớn, 1 đêm). Giá cao thứ hai trong dữ liệu chỉ là 510 và phân vị 99% là 252, nên giá 5.400 gần như chắc chắn là nhập sai. Vì vậy ngưỡng của `adr` được đặt từ 0 đến 1.000.

Có 180 booking không có khách nào, đây là trường hợp không thể xảy ra. `VALID_RANGES` chỉ kiểm tra từng cột riêng lẻ nên mục 6.1 thêm một quy tắc riêng để loại các dòng này. Ngược lại, 715 booking 0 đêm đều có `adr` bằng 0 và chỉ 4,9% bị hủy, có thể là booking trong ngày hoặc phòng miễn phí nên vẫn được giữ lại. 1.959 booking có `adr` bằng 0 cũng được giữ, trong đó 680 booking thuộc phân khúc Complementary (phòng tặng).

Tổng cộng 182 dòng (khoảng 0,2%) sẽ bị loại ở mục 6.1, gồm các dòng có `adr` âm, `adr` trên 1.000 và các dòng không có khách.

### 4.5 Outliers (quy tắc IQR)

Một giá trị được xem là outlier nếu nằm ngoài khoảng $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$ với $IQR = Q_3 - Q_1$. Outlier chưa chắc là lỗi, nên cần xem thêm boxplot và ý nghĩa của từng cột trước khi quyết định.

In [19]:
q1, q3 = raw[NUM_COLS].quantile(0.25), raw[NUM_COLS].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = pd.DataFrame({"ngưỡng dưới": lower, "ngưỡng trên": upper,
                         "số dưới ngưỡng": (raw[NUM_COLS] < lower).sum(), "số trên ngưỡng": (raw[NUM_COLS] > upper).sum()})
outliers["% số dòng"] = (outliers["số dưới ngưỡng"] + outliers["số trên ngưỡng"]) / len(raw) * 100
outliers.sort_values("% số dòng", ascending=False)

,ngưỡng dưới,ngưỡng trên,số dưới ngưỡng,số trên ngưỡng,% số dòng
adults,2.000,2.000,23430,6280,24.885
children,0.000,0.000,0,8590,7.195
required_car_parking_spaces,0.000,0.000,0,7416,6.212
previous_cancellations,0.000,0.000,0,6484,5.431
is_repeated_guest,0.000,0.000,0,3810,3.191
adr,-15.775,211.065,0,3793,3.177
days_in_waiting_list,0.000,0.000,0,3698,3.097
previous_bookings_not_canceled,0.000,0.000,0,3620,3.032
stays_in_week_nights,-2.000,6.000,0,3354,2.809
lead_time,-195.000,373.000,0,3005,2.517


In [20]:
ncols = 3
plot_cols = NUM_COLS[:12]  # giữ hình dễ đọc; nếu nhiều cột hơn thì chọn các cột quan trọng nhất
nrows = max(1, -(-len(plot_cols) // ncols))  # chia làm tròn lên
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Box(y=raw[c], name=c, marker_color=BLUE, boxpoints="outliers", showlegend=False), row=r, col=k)
show_fig(fig, "Spread and outliers of the numeric columns (each panel has its own scale)", height=300 * nrows)

**Nhận xét.** Quy tắc IQR không phù hợp với các cột đếm. Ví dụ ở `adults`, $Q_1 = Q_3 = 2$ nên $IQR = 0$ và mọi booking không phải 2 người lớn đều bị coi là outlier (24,9% số dòng). Tình trạng tương tự xảy ra với `children`, `babies`, `previous_cancellations`, `required_car_parking_spaces` và `days_in_waiting_list`, vì các cột này có $Q_1 = Q_3 = 0$.

Với các cột liên tục, tỷ lệ outlier thấp hơn: `adr` có 3,2% booking trên 211, `lead_time` có 2,5% booking đặt trước hơn 373 ngày và `stays_in_week_nights` có 2,8% booking trên 6 đêm. Những giá trị lớn nhất vẫn có thể giải thích được: 16 booking có hơn 4 người lớn (tối đa 55) đều là booking đoàn và đều bị hủy, 3 booking có 9–10 trẻ em hoặc em bé, 24 booking ở hơn 30 đêm, và có booking nằm trong danh sách chờ tới 391 ngày.

Vì vậy các outlier này được giữ lại, chỉ có `adr` = 5.400 bị xem là lỗi như ở mục 4.4. Để giảm ảnh hưởng của các giá trị lớn, cột số được chuẩn hóa bằng `RobustScaler` (dùng median và IQR) thay vì `StandardScaler`. Mục 6.5 so sánh thêm các cách giữ nguyên, lấy `log1p` và cắt ngưỡng.

### 4.6 Tỷ lệ giữa hai lớp

Khi một lớp chiếm đa số, một mô hình luôn đoán lớp đó cũng đạt accuracy khá cao. Vì vậy cần biết tỷ lệ giữa hai lớp trước khi chọn metric đánh giá.

In [21]:
counts = raw[TARGET].value_counts(dropna=False)
shares = counts / counts.sum() * 100
fig = go.Figure(go.Bar(x=counts.index.astype(str), y=counts.values, marker_color=BLUE,
                       text=[f"{n:,} ({p:.1f}%)" for n, p in zip(counts, shares)], textposition="outside",
                       hovertemplate="%{x}: %{y:,} rows<extra></extra>"))
imbalance_ratio = counts.max() / counts.min()
show_fig(fig, f"Class balance of '{TARGET}': majority/minority ratio = {imbalance_ratio:.1f}",
         xaxis_title=TARGET, yaxis_title="Rows")
print(f"Accuracy khi luôn đoán lớp đa số: {shares.max():.1f}%")

Accuracy khi luôn đoán lớp đa số: 63.0%


**Nhận xét.** Có 75.166 booking không hủy (63,0%) và 44.224 booking bị hủy (37,0%), tỷ lệ khoảng 1,7 : 1. Chỉ cần luôn đoán "không hủy" đã đạt accuracy 63,0%. Sau khi bỏ dòng trùng ở mục 4.2, lớp hủy còn khoảng 27,7%, tỷ lệ khoảng 2,6 : 1.

Do đó các mô hình sẽ được so sánh bằng F1 và PR-AUC của lớp hủy thay vì accuracy. Mục 7 thử thêm `class_weight="balanced"`, SMOTE và điều chỉnh ngưỡng, còn khi chia tập dùng `stratify` để tỷ lệ hủy ở ba tập như nhau.

<a id="s5"></a>
## 5. Phân tích khám phá (EDA)

Phần đầu xem phân phối của từng cột, mối liên hệ với việc hủy phòng và tương quan giữa các cột số. Từ mục 5.6 là ba câu hỏi cụ thể về hành vi hủy phòng.

### 5.1 Phân phối của các cột số

In [22]:
ncols = 3
plot_cols = NUM_COLS[:12]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    fig.add_trace(hist_bar(raw[c], bins=40), row=r, col=k)
fig.update_layout(showlegend=False)
show_fig(fig, "Distribution of the numeric features", height=280 * nrows)
pd.DataFrame({"độ lệch (skewness)": raw[NUM_COLS].skew(),            # |skew| > 1: lệch mạnh
              "% giá trị bằng 0": (raw[NUM_COLS] == 0).mean() * 100,
              "median": raw[NUM_COLS].median(), "max": raw[NUM_COLS].max()}
             ).sort_values("độ lệch (skewness)", key=abs, ascending=False).round(2)

,độ lệch (skewness),% giá trị bằng 0,median,max
babies,24.650,99.230,0.000,10.000
previous_cancellations,24.460,94.570,0.000,26.000
previous_bookings_not_canceled,23.540,96.970,0.000,72.000
adults,18.320,0.340,2.000,55.000
days_in_waiting_list,11.940,96.900,0.000,391.000
adr,10.530,1.640,94.580,"5,400.000"
is_repeated_guest,5.330,96.810,0.000,1.000
required_car_parking_spaces,4.160,93.790,0.000,8.000
children,4.110,92.800,0.000,10.000
stays_in_week_nights,2.860,6.400,2.000,50.000


**Nhận xét.** Phần lớn các cột số lệch phải mạnh và tập trung ở giá trị 0. `babies` bằng 0 ở 99,2% số dòng, `days_in_waiting_list` ở 96,9%, `is_repeated_guest` ở 96,8%, `previous_cancellations` ở 94,6%, `required_car_parking_spaces` ở 93,8% và `children` ở 92,8%. Độ lệch lớn nhất thuộc về `babies` (24,6), `previous_cancellations` (24,5), `previous_bookings_not_canceled` (23,5), `adults` (18,3) và `days_in_waiting_list` (11,9). Độ lệch 10,5 của `adr` chủ yếu do một giá trị 5.400 (mục 4.4).

`lead_time` lệch phải vừa phải (độ lệch 1,35), median 69 ngày và lớn nhất 737 ngày. `arrival_date_week_number` và `arrival_date_day_of_month` phân bố gần như đều.

Vì các cột đếm hầu như chỉ có hai trạng thái bằng 0 và khác 0, mô hình tuyến tính có thể xem chúng gần như các biến nhị phân. Các cột có đuôi dài như `lead_time` và `adr` là lý do dùng `RobustScaler` và thử `log1p` ở mục 6.5.

### 5.2 Phân phối của các cột categorical

In [23]:
TOP_K = 10
ncols = 2
plot_cols = CAT_COLS[:8]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.14)
for i, c in enumerate(plot_cols):
    vc = raw[c].value_counts(dropna=False)
    top = vc.head(TOP_K)
    if len(vc) > TOP_K:  # gộp phần đuôi dài để panel dễ đọc
        top = pd.concat([top, pd.Series({"(other)": vc.iloc[TOP_K:].sum()})])
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Bar(x=top.index.astype(str), y=top.values, marker_color=BLUE, showlegend=False,
                         hovertemplate="%{x}: %{y:,}<extra></extra>"), row=r, col=k)
show_fig(fig, f"Top {TOP_K} categories per categorical feature", height=320 * nrows)

top5 = {c: raw[c].value_counts(normalize=True).head(5) * 100 for c in CAT_COLS}
pd.DataFrame({"5 giá trị phổ biến nhất (% số dòng)": {c: " · ".join(f"{k} {v:.1f}" for k, v in s.items()) for c, s in top5.items()},
              "tổng 5 giá trị (%)": {c: s.sum() for c, s in top5.items()}}).round(1)

,5 giá trị phổ biến nhất (% số dòng),tổng 5 giá trị (%)
hotel,City Hotel 66.4 · Resort Hotel 33.6,100.000
arrival_date_month,August 11.6 · July 10.6 · May 9.9 · October 9....,50.700
meal,BB 77.3 · HB 12.1 · SC 8.9 · Undefined 1.0 · F...,100.000
country,PRT 40.9 · GBR 10.2 · FRA 8.8 · ESP 7.2 · DEU 6.1,73.200
market_segment,Online TA 47.3 · Offline TA/TO 20.3 · Groups 1...,99.200
distribution_channel,TA/TO 82.0 · Direct 12.3 · Corporate 5.6 · GDS...,100.000
reserved_room_type,A 72.0 · D 16.1 · E 5.5 · F 2.4 · G 1.8,97.800
deposit_type,No Deposit 87.6 · Non Refund 12.2 · Refundable...,100.000
agent,9.0 31.0 · 240.0 13.5 · 1.0 7.0 · 14.0 3.5 · 7...,58.500
company,40.0 13.6 · 223.0 11.5 · 67.0 3.9 · 45.0 3.7 ·...,35.900


**Nhận xét.** Nhiều cột có một giá trị chiếm phần lớn: `distribution_channel` là TA/TO ở 82,0% số booking, `deposit_type` là No Deposit ở 87,6%, `meal` là BB ở 77,3%, `reserved_room_type` là A ở 72,0%, `customer_type` là Transient ở 75,1% và City Hotel chiếm 66,4%. `market_segment` phân bố đều hơn với Online TA 47,3%, Offline TA/TO 20,3%, Groups 16,6% và Direct 10,6%.

`country` có 177 giá trị nhưng 5 nước đã chiếm 73,2% số booking, gồm Bồ Đào Nha 40,9%, Anh 10,2%, Pháp 8,8%, Tây Ban Nha 7,2% và Đức 6,1%. Các giá trị hiếm như `Undefined`, `Aviation`, `GDS`, phòng `L`, `P` hay phần lớn mã `agent`, `company` sẽ được gộp lại khi one-hot ở mục 6.3.

### 5.3 Các cột số theo từng lớp

Nếu boxplot của hai lớp ít chồng lên nhau thì cột đó giúp phân biệt booking hủy và không hủy. Bảng bên dưới so sánh median của từng lớp.

In [24]:
ncols = 3
plot_cols = NUM_COLS[:9]
classes = raw[TARGET].dropna().unique()
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.12)
for i, c in enumerate(plot_cols):
    r, k = grid_position(i, ncols)
    for j, cls in enumerate(classes):
        fig.add_trace(go.Box(y=raw.loc[raw[TARGET] == cls, c], name=str(cls), marker_color=QUALITATIVE[j % 6],
                             boxpoints=False, showlegend=(i == 0), legendgroup=str(cls)), row=r, col=k)
show_fig(fig, f"Numeric features by '{TARGET}' class", height=300 * nrows)
display(raw.groupby(TARGET)[NUM_COLS].median().T)  # median theo lớp: chênh lệch lớn = đặc trưng phân biệt tốt
parking = raw["required_car_parking_spaces"] > 0
print(f"Booking có yêu cầu chỗ đỗ xe: {parking.sum():,}, tỷ lệ hủy {raw.loc[parking, TARGET].mean():.1%} "
      f"(không yêu cầu: {raw.loc[~parking, TARGET].mean():.1%})")

is_canceled,0,1
lead_time,45.000,113.000
arrival_date_week_number,28.000,27.000
arrival_date_day_of_month,16.000,16.000
stays_in_weekend_nights,1.000,1.000
stays_in_week_nights,2.000,2.000
adults,2.000,2.000
children,0.000,0.000
babies,0.000,0.000
is_repeated_guest,0.000,0.000
previous_cancellations,0.000,0.000


Booking có yêu cầu chỗ đỗ xe: 7,416, tỷ lệ hủy 0.0% (không yêu cầu: 39.5%)


**Nhận xét.** `lead_time` khác biệt rõ nhất giữa hai lớp: booking không hủy có median 45 ngày, còn booking bị hủy là 113 ngày, gấp khoảng 2,5 lần. `total_of_special_requests` cũng khác nhau, median là 1 ở booking không hủy và 0 ở booking bị hủy. Hai cột này được phân tích thêm ở câu hỏi 1 (5.6) và câu hỏi 3 (5.8).

Với `required_car_parking_spaces`, median của cả hai lớp đều bằng 0, nhưng trong 7.416 booking có yêu cầu chỗ đỗ xe thì không có booking nào bị hủy. Như vậy chỉ nhìn median sẽ bỏ qua thông tin này. `adr` gần như không khác giữa hai lớp (median 92,5 và 96,2 EUR), và các cột đếm còn lại đều có median bằng 0.

### 5.4 Tỷ lệ hủy theo từng giá trị categorical

Với mỗi cột categorical, biểu đồ cho biết tỷ lệ hủy ở các giá trị phổ biến nhất. Đường chấm là tỷ lệ hủy chung của toàn bộ dữ liệu.

In [25]:
overall_rate = (raw[TARGET] == MINORITY).mean() * 100
ncols = 2
plot_cols = CAT_COLS[:8]
nrows = max(1, -(-len(plot_cols) // ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=plot_cols, vertical_spacing=0.14)
for i, c in enumerate(plot_cols):
    top_categories = raw[c].value_counts().head(10).index
    rate = (raw[raw[c].isin(top_categories)].groupby(c)[TARGET]
            .apply(lambda s: (s == MINORITY).mean() * 100).sort_values())
    r, k = grid_position(i, ncols)
    fig.add_trace(go.Bar(x=rate.index.astype(str), y=rate.values, marker_color=ORANGE, showlegend=False,
                         hovertemplate="%{x}: %{y:.1f}%<extra></extra>"), row=r, col=k)
    fig.add_hline(y=overall_rate, line_dash="dot", line_color=MUTED, row=r, col=k)
show_fig(fig, f"Share of '{MINORITY}' per category (dotted line = overall {overall_rate:.1f}%)", height=320 * nrows)

rate_rows = {}
for c in plot_cols:
    top_categories = raw[c].value_counts().head(10).index
    rate = (raw[raw[c].isin(top_categories)].groupby(c)[TARGET]
            .apply(lambda s: (s == MINORITY).mean() * 100).sort_values(ascending=False))
    rate_rows[c] = " · ".join(f"{k} {v:.1f}" for k, v in rate.items())
pd.Series(rate_rows, name=f"% '{MINORITY}' theo category (giảm dần)").to_frame()

,% '1' theo category (giảm dần)
hotel,City Hotel 41.7 · Resort Hotel 27.8
arrival_date_month,June 41.5 · April 40.8 · May 39.7 · September ...
meal,FB 59.9 · BB 37.4 · SC 37.2 · HB 34.5 · Undefi...
country,PRT 56.6 · BRA 37.3 · ITA 35.4 · ESP 25.4 · IR...
market_segment,Undefined 100.0 · Groups 61.1 · Online TA 36.7...
distribution_channel,Undefined 80.0 · TA/TO 41.0 · Corporate 22.1 ·...
reserved_room_type,P 100.0 · H 40.8 · A 39.1 · G 36.4 · L 33.3 · ...
deposit_type,Non Refund 99.4 · No Deposit 28.4 · Refundable...


**Nhận xét.** Các tỷ lệ ở đây được tính trên toàn bộ dữ liệu, với tỷ lệ hủy chung là 37,0%. `deposit_type` có chênh lệch lớn nhất: booking Non Refund bị hủy 99,4%, trong khi No Deposit là 28,4% và Refundable là 22,2%. Kết quả này ngược với suy nghĩ thông thường và được tìm hiểu ở câu hỏi 2 (5.7).

Theo `market_segment`, nhóm Groups hủy 61,1%, cao hơn nhiều so với Direct (15,3%), Corporate (18,7%) và Complementary (13,1%). Theo `distribution_channel`, TA/TO hủy 41,0% còn Direct chỉ 17,5%. Khách Bồ Đào Nha hủy 56,6%, gấp khoảng ba lần khách Đức (16,7%), Pháp (18,6%) và Anh (20,2%). City Hotel hủy 41,7% so với 27,8% ở Resort Hotel. Tỷ lệ hủy theo tháng chỉ dao động trong khoảng 31,2% (tháng 11) đến 41,5% (tháng 6).

Do các dòng trùng ở mục 4.2 tập trung ở nhóm Groups, Non Refund và khách Bồ Đào Nha, một số tỷ lệ cao ở trên bị đẩy lên. Vì vậy các câu hỏi 5.6–5.8 tính trên cả toàn bộ dữ liệu và dữ liệu đã bỏ trùng.

### 5.5 Tương quan giữa các cột số

Hệ số Pearson đo mức độ liên hệ tuyến tính giữa hai cột. Nếu hai cột có |r| > 0,9 thì chúng gần như trùng thông tin và chỉ cần giữ một cột.

In [26]:
corr = raw[NUM_COLS].corr()
fig = go.Figure(go.Heatmap(z=corr.values, x=corr.columns, y=corr.columns, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=corr.round(2).values, texttemplate="%{text}", xgap=2, ygap=2,
                           colorbar=dict(title="r"), hovertemplate="%{y} x %{x}: %{z:.2f}<extra></extra>"))
show_fig(fig, "Pearson correlation between numeric features", height=max(420, 40 * len(NUM_COLS)),
         yaxis_autorange="reversed")

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()  # tam giác trên: mỗi cặp một lần
pairs[pairs.abs() > 0.9].rename("r").to_frame() if (pairs.abs() > 0.9).any() else print("Không có cặp nào có |r| > 0,9.")

Không có cặp nào có |r| > 0,9.


**Nhận xét.** Không có cặp nào có |r| > 0,9 nên không cần bỏ cột nào. Các cặp có tương quan cao nhất đều dễ giải thích: số đêm cuối tuần và số đêm trong tuần (r = 0,50) vì kỳ nghỉ dài thường có cả hai, `is_repeated_guest` và `previous_bookings_not_canceled` (r = 0,42) vì khách quen là người đã từng ở, `children` và `adr` (r = 0,32) vì thêm người thì giá phòng cao hơn.

### 5.6 Câu hỏi 1: Đặt phòng càng sớm thì càng dễ hủy?

Ở mục 5.3, `lead_time` là cột có median chênh lệch nhiều nhất giữa hai lớp. Câu hỏi đặt ra là tỷ lệ hủy thay đổi thế nào theo thời gian đặt trước và có khác nhau giữa hai khách sạn hay không.

Vì phần lớn dòng trùng là booking bị hủy (mục 4.2), tỷ lệ hủy được tính trên hai tập: toàn bộ dữ liệu (đường chấm) và dữ liệu đã bỏ trùng (đường liền). Tập thứ hai cũng là dữ liệu dùng để huấn luyện mô hình sau mục 6.1.

In [27]:
# Dòng không trùng: bỏ các cột trong DROP_COLS rồi bỏ bản sao, giống bước làm sạch ở 6.1
unique_rows = raw.drop(columns=DROP_COLS).drop_duplicates()
print(f"Toàn bộ dòng: {len(raw):,} (hủy {raw[TARGET].mean():.1%}) | dòng không trùng: {len(unique_rows):,} "
      f"(hủy {unique_rows[TARGET].mean():.1%})")

LEAD_BINS = [-1, 7, 30, 90, 180, 365, np.inf]          # mốc theo ngày: 1 tuần, 1 tháng, 1 quý, nửa năm, 1 năm
LEAD_LABELS = ["0–7", "8–30", "31–90", "91–180", "181–365", ">365"]


def cancel_by_lead(d):
    """Tỷ lệ hủy (%) và số booking theo khách sạn × nhóm lead_time."""
    return (d.assign(lead_group=pd.cut(d["lead_time"], LEAD_BINS, labels=LEAD_LABELS))
             .groupby(["hotel", "lead_group"], observed=True)[TARGET].agg(rate="mean", n="size")
             .assign(rate=lambda t: t["rate"] * 100))


q1 = pd.concat({"all rows": cancel_by_lead(raw), "unique rows": cancel_by_lead(unique_rows)},
               names=["data"]).reset_index()
q1_u = q1[q1["data"] == "unique rows"].set_index(["hotel", "lead_group"])["rate"]

fig = go.Figure()
for (data, hotel), part in q1.groupby(["data", "hotel"]):
    fig.add_trace(go.Scatter(
        x=part["lead_group"].astype(str), y=part["rate"], mode="lines+markers", name=f"{hotel} ({data})",
        line=dict(color=BLUE if hotel == "City Hotel" else ORANGE, dash="dot" if data == "all rows" else "solid"),
        customdata=part["n"], hovertemplate="%{x} days: %{y:.1f}% of %{customdata:,} bookings<extra></extra>"))
show_fig(fig, f"Cancellations rise with lead time: City Hotel {q1_u['City Hotel', '0–7']:.0f}% (≤7 days) "
              f"→ {q1_u['City Hotel', '181–365']:.0f}% (181–365 days) on unique rows",
         xaxis_title="Lead time (days between booking and arrival)", yaxis_title="Cancelled bookings (%)",
         yaxis_rangemode="tozero", height=460)
display(q1.pivot_table(index="lead_group", columns=["data", "hotel"], values="rate", observed=True)
          .round(1).rename_axis(index="% bị hủy"))
q1.pivot_table(index="lead_group", columns=["data", "hotel"], values="n", aggfunc="sum", observed=True
               ).rename_axis(index="số booking")

Toàn bộ dòng: 119,390 (hủy 37.0%) | dòng không trùng: 85,340 (hủy 27.7%)


data       all rows              unique rows             
hotel    City Hotel Resort Hotel  City Hotel Resort Hotel
% bị hủy                                                 
0–7          12.200        6.500      10.500        6.000
8–30         30.900       21.900      28.000       20.900
31–90        39.900       32.400      33.500       29.800
91–180       48.000       37.400      36.900       32.300
181–365      62.700       41.300      45.700       35.200
>365         71.100       46.600      53.000       21.600

data         all rows              unique rows             
hotel      City Hotel Resort Hotel  City Hotel Resort Hotel
số booking                                                 
0–7             10808         8938        9775         8312
8–30            12554         6406       10360         5743
31–90           20797         8756       14791         7428
91–180          18223         8216       11260         6535
181–365         14244         7300        5583         5053
>365             2704          444         338          162

**Kết luận.** Tỷ lệ hủy tăng dần theo thời gian đặt trước ở cả hai khách sạn. Trên dữ liệu đã bỏ trùng, ở City Hotel tỷ lệ hủy là 10,5% với booking đặt trước tối đa 7 ngày, sau đó tăng lên 28,0%, 33,5%, 36,9% và 45,7% ở nhóm 181–365 ngày, rồi 53,0% với booking đặt trước hơn một năm, tức là gấp khoảng 4–5 lần. Ở Resort Hotel tỷ lệ tăng từ 6,0% lên 35,2% (181–365 ngày) rồi giảm còn 21,6% ở nhóm trên 365 ngày, nhưng nhóm này chỉ có 162 booking nên chưa đủ tin cậy. Ở cùng một nhóm `lead_time`, City Hotel luôn có tỷ lệ hủy cao hơn Resort Hotel.

Dòng trùng làm tỷ lệ hủy ở các nhóm đặt sớm cao lên đáng kể: nhóm 181–365 ngày ở City Hotel là 62,7% nếu tính trên toàn bộ dữ liệu, so với 45,7% sau khi bỏ trùng; nhóm trên 365 ngày ở Resort Hotel là 46,6% so với 21,6%.

Như vậy `lead_time` có liên hệ rõ với việc hủy phòng, nhưng mối liên hệ này không tuyến tính và khác nhau giữa hai khách sạn. Đây là lý do các mô hình dạng cây được kỳ vọng cho kết quả tốt hơn mô hình tuyến tính.

### 5.7 Câu hỏi 2: Vì sao booking Non Refund lại bị hủy nhiều nhất?

Thông thường khách đã trả tiền không hoàn lại sẽ ít hủy hơn, nhưng mục 5.4 cho thấy điều ngược lại. Phần này xem tỷ lệ hủy theo `deposit_type` có bị ảnh hưởng bởi dòng trùng hay không và các booking Non Refund thuộc nhóm khách nào.

In [28]:
def cancel_by(d, col):
    """Tỷ lệ hủy (%) và số booking theo một cột."""
    return d.groupby(col)[TARGET].agg(rate="mean", n="size").assign(rate=lambda t: t["rate"] * 100)


q2 = pd.concat({"all rows": cancel_by(raw, "deposit_type"), "unique rows": cancel_by(unique_rows, "deposit_type")},
               names=["data"]).reset_index()
nr_raw = (raw["deposit_type"] == "Non Refund").sum()
nr_unique = (unique_rows["deposit_type"] == "Non Refund").sum()
nr_rate = q2.set_index(["data", "deposit_type"]).loc[("unique rows", "Non Refund"), "rate"]

fig = px.bar(q2, x="deposit_type", y="rate", color="data", barmode="group",
             text=[f"{r:.1f}%<br>n={n:,}" for r, n in zip(q2["rate"], q2["n"])],
             color_discrete_sequence=[MUTED, BLUE])
fig.update_traces(textposition="outside")
show_fig(fig, f"Non Refund is still cancelled {nr_rate:.0f}% on unique rows, but only {nr_unique:,} of "
              f"{nr_raw:,} Non Refund rows are not duplicates",
         xaxis_title="Deposit type", yaxis_title="Cancelled bookings (%)", yaxis_range=[0, 115], height=460, legend_title_text="")

# Booking Non Refund (không trùng) khác booking No Deposit ở điểm nào?
profile = {}
for dep in ["Non Refund", "No Deposit"]:
    d = unique_rows[unique_rows["deposit_type"] == dep]
    profile[dep] = {"số booking": len(d),
                    "% Groups": (d["market_segment"] == "Groups").mean() * 100,
                    "% Offline TA/TO": (d["market_segment"] == "Offline TA/TO").mean() * 100,
                    "% khách Bồ Đào Nha (PRT)": (d["country"] == "PRT").mean() * 100,
                    "% City Hotel": (d["hotel"] == "City Hotel").mean() * 100,
                    "median lead_time (ngày)": d["lead_time"].median()}
pd.DataFrame(profile).round(1)

,Non Refund,No Deposit
số booking,992.000,"84,259.000"
% Groups,62.800,4.100
% Offline TA/TO,28.200,15.400
% khách Bồ Đào Nha (PRT),94.900,30.500
% City Hotel,82.500,60.900
median lead_time (ngày),192.000,47.000


**Kết luận.** Trên toàn bộ dữ liệu, booking Non Refund bị hủy 99,4% (14.587 dòng). Sau khi bỏ trùng, tỷ lệ này vẫn là 95,0%, nhưng chỉ còn 992 booking vì có tới 13.595 dòng Non Refund (93,2%) là bản sao. Như vậy tỷ lệ hủy cao là có thật nhưng chỉ dựa trên một số ít booking.

Các booking Non Refund cũng khác hẳn booking không đặt cọc. 62,8% thuộc phân khúc Groups và 28,2% thuộc Offline TA/TO (so với 4,1% và 15,4% ở No Deposit), 94,9% là khách Bồ Đào Nha (so với 30,5%), 82,5% đặt ở City Hotel (so với 60,9%) và median thời gian đặt trước là 192 ngày (so với 47 ngày). Từ các đặc điểm này, có thể đoán phần lớn booking Non Refund là các lô phòng do đoàn hoặc hãng lữ hành trong nước giữ từ sớm và trả lại khi không bán hết. Khi đó điều khoản không hoàn tiền nằm trong hợp đồng với đại lý chứ không phải khách lẻ đã thanh toán. Đây chỉ là giả thuyết vì dữ liệu không có cột nào xác nhận trực tiếp.

`deposit_type` vì vậy vẫn là một đặc trưng có ích nhưng chỉ dựa trên khoảng 1.000 booking. Nếu không bỏ trùng, 13.595 bản sao sẽ nằm ở cả train và test, làm kết quả trên test cao hơn thực tế.

### 5.8 Câu hỏi 3: Yêu cầu đặc biệt và khách quen có làm giảm hủy phòng?

Mục 5.3 cho thấy booking không hủy thường có nhiều yêu cầu đặc biệt hơn (giường phụ, tầng cao, ...). Khách đưa ra yêu cầu đặc biệt thường đã có kế hoạch rõ ràng cho chuyến đi. Phần này xem tỷ lệ hủy giảm bao nhiêu khi có yêu cầu đặc biệt và điều đó có đúng với cả khách quen hay không.

In [29]:
q3 = (unique_rows
      .assign(requests=unique_rows["total_of_special_requests"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"}),
              guest=unique_rows["is_repeated_guest"].map({0: "New guest", 1: "Repeated guest"}))
      .groupby(["guest", "requests"])[TARGET].agg(rate="mean", n="size").reset_index())
q3["rate"] *= 100
r3 = q3.set_index(["guest", "requests"])["rate"]

fig = px.bar(q3, x="requests", y="rate", color="guest", barmode="group",
             text=[f"{r:.1f}%<br>n={n:,}" for r, n in zip(q3["rate"], q3["n"])],
             color_discrete_sequence=[BLUE, AQUA])
fig.update_traces(textposition="outside")
show_fig(fig, f"New guests with no special request cancel {r3['New guest', '0']:.0f}%, "
              f"repeated guests with one request only {r3['Repeated guest', '1']:.0f}% (unique rows)",
         xaxis_title="Number of special requests", yaxis_title="Cancelled bookings (%)", yaxis_range=[0, 45], height=460, legend_title_text="")

rep = unique_rows.groupby("is_repeated_guest")[TARGET].mean() * 100
print(f"Tỷ lệ hủy: khách mới {rep[0]:.1f}% | khách quen {rep[1]:.1f}% (dòng không trùng)")

# Cùng câu hỏi trên toàn bộ dòng: dòng trùng làm lệch bao nhiêu?
pd.DataFrame({
    "toàn bộ dòng": raw.groupby(raw["total_of_special_requests"].clip(upper=3))[TARGET].mean() * 100,
    "dòng không trùng": unique_rows.groupby(unique_rows["total_of_special_requests"].clip(upper=3))[TARGET].mean() * 100,
}).rename_axis("số yêu cầu đặc biệt (3 = 3+)").round(1)

Tỷ lệ hủy: khách mới 28.5% | khách quen 7.7% (dòng không trùng)


,toàn bộ dòng,dòng không trùng
số yêu cầu đặc biệt (3 = 3+),,
0,47.700,33.600
1,22.000,22.500
2,22.100,21.200
3,16.800,16.300


**Kết luận.** Cả số yêu cầu đặc biệt và việc là khách quen đều làm tỷ lệ hủy giảm rõ. Với khách mới, tỷ lệ hủy là 34,7% khi không có yêu cầu nào, giảm còn 23,2% khi có 1 yêu cầu, 21,7% với 2 yêu cầu và 16,9% với từ 3 yêu cầu trở lên. Chỉ một yêu cầu đã làm tỷ lệ hủy giảm khoảng một phần ba.

Khách quen hủy 7,7%, chỉ bằng khoảng một phần tư so với 28,5% ở khách mới. Ở từng mức yêu cầu, khách quen vẫn hủy ít hơn: 10,0% so với 34,7% khi không có yêu cầu, 4,1% so với 23,2% khi có 1 yêu cầu, 6,0% so với 21,7% khi có 2 yêu cầu và 7,5% so với 16,9% khi có từ 3 yêu cầu. Do số khách quen ở mỗi nhóm khá nhỏ (173–1.829 booking), chênh lệch giữa các mức yêu cầu của riêng nhóm khách quen chưa đáng tin.

Dòng trùng chỉ ảnh hưởng đến nhóm không có yêu cầu đặc biệt (47,7% trên toàn bộ dữ liệu so với 33,6% sau khi bỏ trùng). Các nhóm có từ 1 yêu cầu trở lên gần như không thay đổi (ví dụ 22,0% và 22,5%), cho thấy các bản sao hầu hết là booking không có yêu cầu đặc biệt. `total_of_special_requests` và `is_repeated_guest` do đó là hai đặc trưng có ích, bổ sung cho `lead_time` và `deposit_type`.

<a id="s6"></a>
## 6. Chuẩn bị dữ liệu

Các bước xử lý theo từng dòng (chuẩn hóa chữ, bỏ dòng trùng, bỏ giá trị vô lý) được làm trước khi chia tập vì chúng không dùng thông tin của các dòng khác. Các bước cần tính thống kê như median để điền missing, tham số để chuẩn hóa hay danh sách giá trị cho one-hot chỉ được tính trên tập train, sau đó áp dụng cho validation và test. Nếu tính trên toàn bộ dữ liệu, thông tin của validation và test sẽ lọt vào quá trình huấn luyện.

### 6.1 Làm sạch dữ liệu

In [30]:
def normalize_values(df):
    """Sửa chữ theo từng dòng, dùng chung cho dữ liệu huấn luyện và dữ liệu mới (mục 9)."""
    out = df.copy()
    for c in out.columns.difference(out.select_dtypes("number").columns):
        s = out[c]
        s = s.where(s.isna(), s.astype(str).str.strip())   # chỉ strip giá trị thật; NaN giữ nguyên
        out[c] = s.replace(NA_TOKENS, np.nan)              # chuỗi missing ẩn -> NaN
    for c, mapping in SPELLING_MAPS.items():
        lowered = out[c].str.lower()
        out[c] = lowered.map(mapping).fillna(out[c])       # biến thể có trong map -> dạng chuẩn; còn lại giữ nguyên
    for c in TO_NUMERIC:
        out[c] = pd.to_numeric(out[c].astype(str).str.replace(",", ""), errors="coerce")
    return out


def add_features(df):
    """Đặc trưng mới tính từ chính dòng đó (không dùng thống kê), dùng chung cho dữ liệu mới (mục 9)."""
    out = df.copy()
    out["total_nights"] = out["stays_in_weekend_nights"] + out["stays_in_week_nights"]
    out["total_guests"] = out["adults"] + out["children"].fillna(0) + out["babies"]
    for c in ["agent", "company"]:
        out[f"has_{c}"] = out[c].notna().astype(int)       # thiếu = không qua đại lý / công ty (mục 4.1)
    for c in FORCE_CATEGORICAL:                             # mã số -> chữ, để bộ mã hóa coi là category
        out[c] = pd.to_numeric(out[c], errors="coerce").map(lambda v: f"{v:.0f}", na_action="ignore")
    return out


def clean_table(df):
    """Áp mọi quy tắc an toàn trước khi chia và ghi lại số dòng sau mỗi bước."""
    steps = [("dữ liệu gốc", len(df))]
    out = normalize_values(df)
    out = out.drop(columns=DROP_COLS)
    out = out.dropna(subset=[TARGET])                      # dòng không có nhãn không dùng được để huấn luyện hay kiểm tra
    steps.append(("có nhãn", len(out)))
    out = out.drop_duplicates()
    steps.append(("bỏ dòng trùng hoàn toàn", len(out)))
    valid = pd.Series(True, index=out.index)
    for c, (low, high) in VALID_RANGES.items():
        ok = out[c].isna()                                 # giá trị thiếu không phải vô lý: imputer sẽ xử lý
        ok |= out[c].between(-np.inf if low is None else low, np.inf if high is None else high)
        valid &= ok
    out = out.loc[valid]
    steps.append(("bỏ giá trị vô lý (VALID_RANGES)", len(out)))
    for name, rule in ROW_RULES.items():                   # quy tắc trên nhiều cột
        out = out.loc[rule(out)]
        steps.append((f"bỏ dòng vi phạm: {name}", len(out)))
    out = add_features(out)
    raw_index = out.index.to_numpy()                        # vị trí các dòng giữ lại trong raw, dùng ở mục 7.2
    return out.reset_index(drop=True), pd.DataFrame(steps, columns=["bước", "số dòng"]), raw_index


df, clean_steps, raw_index = clean_table(raw)
FEATURES, NUM_COLS, CAT_COLS = column_roles(df)  # kiểu dữ liệu có thể đã đổi (TO_NUMERIC), nên tính lại vai trò cột
print("Numeric:", NUM_COLS)
print("Categorical:", CAT_COLS)
print(f"Tỷ lệ hủy sau làm sạch: {df[TARGET].mean():.1%}")
clean_steps.assign(**{"bị bỏ": lambda t: t["số dòng"].shift(1) - t["số dòng"]})

Numeric: ['lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'has_agent', 'has_company']
Categorical: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'agent', 'company', 'customer_type']
Tỷ lệ hủy sau làm sạch: 27.7%


,bước,số dòng,bị bỏ
0,dữ liệu gốc,119390,NaN
1,có nhãn,119390,0.000
2,bỏ dòng trùng hoàn toàn,85340,"34,050.000"
3,bỏ giá trị vô lý (VALID_RANGES),85338,2.000
4,bỏ dòng vi phạm: có ít nhất 1 khách,85175,163.000


**Nhận xét.** Sau khi làm sạch còn 85.175 trong 119.390 dòng (71,3%), với tỷ lệ hủy 27,7%. Có 34.050 dòng trùng bị bỏ; con số này lớn hơn 31.994 ở mục 4.2 vì việc so trùng được thực hiện sau khi đã bỏ các cột trong `DROP_COLS`. Ngoài ra có 2 dòng `adr` sai (âm và 5.400) và 163 dòng không có khách bị loại; số dòng không có khách ít hơn 180 ở mục 4.4 vì một phần đã bị bỏ ở bước trùng.

Các cột trong `DROP_COLS` bị loại vì những lý do sau: `reservation_status` và `reservation_status_date` chính là kết quả hủy hay không và ngày ghi kết quả đó, `assigned_room_type` chỉ có lúc check-in, `booking_changes` đếm số lần sửa booking đến lúc check-in hoặc hủy, còn `arrival_date_year` chỉ gồm 2015–2017 nên không dùng được cho các năm sau.

Hàm `add_features` tạo thêm `total_nights`, `total_guests` và hai cột cờ `has_agent`, `has_company` để giữ lại thông tin về ô thiếu ở mục 4.1. Mã `agent` và `company` được chuyển sang dạng chữ để mã hóa như categorical. Hàm này cũng được dùng lại cho dữ liệu mới ở mục 9.

### 6.2 Chia train / validation / test

Dữ liệu được chia thành 70% train để huấn luyện, 15% validation để so sánh và chọn mô hình, 15% test để đánh giá mô hình cuối cùng một lần duy nhất ở mục 8. Tham số `stratify` giữ tỷ lệ hủy như nhau ở cả ba tập, `random_state=SEED` giúp lần chạy sau chia ra đúng như cũ.

In [31]:
X = df[FEATURES]
MINORITY = df[TARGET].value_counts().idxmin()      # tính lại: làm sạch có thể đã đổi cách viết nhãn
label_encoder = LabelEncoder().fit(df[TARGET])
y = label_encoder.transform(df[TARGET])            # tên lớp -> 0..K-1, theo thứ tự bảng chữ cái
CLASS_NAMES = [str(c) for c in label_encoder.classes_]
POS = CLASS_NAMES.index(str(MINORITY))             # chỉ số của lớp thiểu số (lớp "dương" trong bài toán nhị phân)
IS_BINARY = len(CLASS_NAMES) == 2

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=TEST_SIZE + VAL_SIZE, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE), stratify=y_temp, random_state=SEED)

split_table = pd.DataFrame({
    "tập": ["train", "validation", "test"],
    "số dòng": [len(X_train), len(X_val), len(X_test)],
    "tỷ lệ": [len(p) / len(X) for p in (X_train, X_val, X_test)],
    f"% '{MINORITY}'": [np.mean(p == POS) * 100 for p in (y_train, y_val, y_test)],
})
split_table

,tập,số dòng,tỷ lệ,% '1'
0,train,59622,0.700,27.684
1,validation,12776,0.150,27.685
2,test,12777,0.150,27.683


### 6.3 Encoding và scaling

| Nhóm cột | Điền missing | Biến đổi | Lý do |
|---|---|---|---|
| numeric (`num`) | median | `RobustScaler`: $(x - \text{median}) / IQR$ | Nhiều cột lệch phải (mục 5.1), median và IQR ít bị ảnh hưởng bởi outlier hơn mean và độ lệch chuẩn |
| numeric có đuôi dài (`tail`, gồm các cột trong `OUTLIER_COLS`) | median | giữ nguyên, `log1p` hoặc cắt ngưỡng (chọn ở 6.5), sau đó `RobustScaler` | Chỉ xử lý outlier ở cột liên tục, vì mục 4.5 cho thấy cột đếm có IQR = 0 |
| nominal (`cat`) | giá trị `"missing"` (chọn ở 6.4) | `OneHotEncoder`, gộp các giá trị chiếm dưới 1% train, giá trị mới chưa gặp được xếp vào nhóm hiếm | Các giá trị không có thứ tự, nếu đánh số 0, 1, 2 mô hình sẽ hiểu nhầm là có lớn nhỏ |
| ordinal (`ord`, tháng đến) | giá trị phổ biến nhất | `OrdinalEncoder` theo thứ tự tháng, sau đó `StandardScaler` | Tháng có thứ tự tự nhiên nên chỉ cần một cột thay vì 12 cột one-hot |

Hàm `make_preprocessor` tạo `ColumnTransformer` theo các lựa chọn trên để mục 6.4–6.5 so sánh các cách xử lý. Bộ tiền xử lý được đặt trong pipeline của từng mô hình nên các thống kê chỉ được tính trên dữ liệu train, kể cả khi chạy cross-validation.

In [32]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Cắt giá trị về [phân vị low, phân vị high]; ngưỡng học trên dữ liệu được fit (train)."""

    def __init__(self, low=0.01, high=0.99):
        self.low, self.high = low, high

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(X, self.low, axis=0)
        self.upper_ = np.nanquantile(X, self.high, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


def make_preprocessor(num_impute="median", cat_impute="constant", outlier="none", drop=()):
    """ColumnTransformer theo tùy chọn: cách điền missing (6.4), cách xử lý outlier (6.5), cột bỏ đi."""
    num = [c for c in NUM_COLS if c not in drop]
    tail = [c for c in OUTLIER_COLS if c in num]
    rest = [c for c in num if c not in tail]
    nominal = [c for c in CAT_COLS if c not in ORDINAL_COLS and c not in drop]
    ordinal = [c for c in ORDINAL_COLS if c in CAT_COLS]

    num_imputer = KNNImputer(n_neighbors=5) if num_impute == "knn" else SimpleImputer(strategy="median")
    tail_steps = [("imputer", clone(num_imputer))]
    if outlier == "clip":
        tail_steps.append(("clip", QuantileClipper(*CLIP_QUANTILES)))
    elif outlier == "log":
        tail_steps.append(("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")))
    tail_steps.append(("scaler", RobustScaler()))
    cat_imputer = (SimpleImputer(strategy="constant", fill_value="missing") if cat_impute == "constant"
                   else SimpleImputer(strategy="most_frequent"))
    return ColumnTransformer([
        ("num", Pipeline([("imputer", clone(num_imputer)), ("scaler", RobustScaler())]), rest),
        ("tail", Pipeline(tail_steps), tail),
        ("cat", Pipeline([("imputer", cat_imputer),
                          ("encoder", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                    min_frequency=MIN_CATEGORY_FREQ, sparse_output=False))]), nominal),
        ("ord", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("encoder", OrdinalEncoder(categories=[ORDINAL_COLS[c] for c in ordinal])),
                          ("scaler", StandardScaler())]), ordinal),
    ])


preprocessor = make_preprocessor()  # mặc định: missing = category riêng, chưa xử lý outlier; chọn lại ở cuối 6.5

# Fit một lần ở đây chỉ để kiểm tra kết quả; các mô hình bên dưới tự fit bản sao của riêng chúng.
prep_check = clone(preprocessor).fit(X_train)                 # học median, IQR, danh sách category
X_train_ml = prep_check.transform(X_train)
X_val_ml, X_test_ml = prep_check.transform(X_val), prep_check.transform(X_test)
feature_names = prep_check.get_feature_names_out()

print(f"train {X_train_ml.shape} | validation {X_val_ml.shape} | test {X_test_ml.shape}")
print(f"{len(NUM_COLS)} cột numeric + {len(CAT_COLS)} cột categorical -> {len(feature_names)} cột cho mô hình")
assert not np.isnan(X_train_ml).any() and not np.isnan(X_test_ml).any(), "Còn NaN sau tiền xử lý"
pd.Series([n.split("__")[0] for n in feature_names]).value_counts().rename("số cột sau biến đổi").to_frame()

train (59622, 81) | validation (12776, 81) | test (12777, 81)
19 cột numeric + 11 cột categorical -> 81 cột cho mô hình


,số cột sau biến đổi
cat,61
num,13
tail,6
ord,1


### 6.4 So sánh các cách xử lý missing values

Mục 4.1 cho thấy việc thiếu `agent`, `company` và `country` có liên quan đến tỷ lệ hủy. Bốn cách xử lý được so sánh trên cùng một mô hình là Logistic Regression (`class_weight="balanced"`), đánh giá trên tập validation:

| Cách | Mô tả |
|---|---|
| (a) Bỏ dòng | Chỉ huấn luyện trên các dòng train không thiếu ô nào; tập validation giữ nguyên và được điền bằng giá trị phổ biến nhất để dự đoán |
| (b) Bỏ cột, điền median/mode | Bỏ `company` (thiếu hơn 50%) và các cột cờ `has_*`, điền median cho cột số và giá trị phổ biến nhất cho cột chữ |
| (c) Coi missing là một giá trị | Ô thiếu được gán giá trị `"missing"`, giữ hai cột `has_agent` và `has_company` |
| (d) KNN | Giống (c) nhưng cột số được điền bằng `KNNImputer` dựa trên 5 dòng gần nhất |

In [33]:
def quick_scores(prep, X_tr=None, y_tr=None):
    """Fit Logistic Regression (balanced) với một bộ tiền xử lý, trả về metric của lớp hủy trên validation."""
    X_tr = X_train if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    model = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))])
    model.fit(X_tr, y_tr)
    proba, pred = model.predict_proba(X_val)[:, POS], model.predict(X_val)
    return {"F1 (hủy)": f1_score(y_val == POS, pred == POS), "recall (hủy)": recall_score(y_val == POS, pred == POS),
            "PR-AUC": average_precision_score(y_val == POS, proba), "ROC-AUC": roc_auc_score(y_val == POS, proba),
            "số dòng train": len(X_tr)}


complete = X_train.notna().all(axis=1).to_numpy()
missing_results = pd.DataFrame({
    "(a) bỏ dòng thiếu": quick_scores(make_preprocessor(cat_impute="most_frequent"), X_train[complete], y_train[complete]),
    "(b) bỏ cột + median/mode": quick_scores(make_preprocessor(cat_impute="most_frequent",
                                                               drop=("company", "has_company", "has_agent"))),
    "(c) missing = category": quick_scores(make_preprocessor(cat_impute="constant")),
    "(d) KNN + category": quick_scores(make_preprocessor(num_impute="knn", cat_impute="constant")),
}).T
others = missing_results.iloc[1:]                     # (b), (c), (d): đều dùng đủ dòng train
spread = others["PR-AUC"].max() - others["PR-AUC"].min()

plot_df = (missing_results.drop(columns="số dòng train").reset_index(names="strategy")
           .melt(id_vars="strategy", var_name="metric", value_name="score"))
plot_df = plot_df.replace(LABEL_EN)
fig = px.bar(plot_df, x="metric", y="score", color="strategy", barmode="group",
             text=[f"{v:.3f}" for v in plot_df["score"]], color_discrete_sequence=QUALITATIVE)
fig.update_traces(textposition="outside")
show_fig(fig, f"Dropping incomplete rows leaves {int(missing_results.iloc[0]['số dòng train']):,} training rows "
              f"(PR-AUC {missing_results.iloc[0]['PR-AUC']:.2f}); the other strategies differ by {spread:.3f}",
         xaxis_title="Metric on validation (cancelled class)", yaxis_title="Score", yaxis_range=[0, 1.05],
         height=460, legend_title_text="")
missing_results.round(4)

,F1 (hủy),recall (hủy),PR-AUC,ROC-AUC,số dòng train
(a) bỏ dòng thiếu,0.184,0.137,0.289,0.524,122.000
(b) bỏ cột + median/mode,0.645,0.803,0.655,0.846,"59,622.000"
(c) missing = category,0.648,0.805,0.654,0.849,"59,622.000"
(d) KNN + category,0.648,0.805,0.654,0.849,"59,622.000"


**Nhận xét.** Cách (a) gần như không dùng được: do hầu hết các dòng đều thiếu `company`, chỉ còn 122 trong 59.622 dòng train (0,2%). Mô hình học từ 122 dòng chỉ đạt PR-AUC 0,289, gần bằng tỷ lệ hủy 27,7% của một mô hình đoán ngẫu nhiên.

Ba cách còn lại cho kết quả gần như nhau, với PR-AUC từ 0,654 đến 0,655, F1 lớp hủy từ 0,645 đến 0,647 và ROC-AUC từ 0,846 đến 0,849. Logistic Regression vẫn bù được phần thông tin khi bỏ cột `company` nhờ các cột khác. Cách (d) cho kết quả trùng với (c) vì trong các cột số chỉ có 4 ô `children` bị thiếu.

Cách (c) được chọn vì kết quả ngang với (b), ROC-AUC cao hơn một chút (0,849 so với 0,846), không phải bỏ cột nào và giữ được ý nghĩa của ô trống là booking không qua đại lý hay công ty. Các mô hình dạng cây ở mục 7 có thể tận dụng giá trị `"missing"` này tốt hơn mô hình tuyến tính.

### 6.5 So sánh các cách xử lý outlier

Ba cách được so sánh trên các cột liên tục có đuôi dài (`OUTLIER_COLS`), dùng cách xử lý missing (c) và cùng mô hình như ở 6.4:

| Cách | Mô tả |
|---|---|
| Giữ nguyên | Chỉ chuẩn hóa bằng `RobustScaler` |
| `log1p` | Lấy $\log(1 + x)$ để nén các giá trị lớn trước khi chuẩn hóa (các cột này đều không âm sau khi làm sạch) |
| Cắt ngưỡng | Giá trị dưới phân vị 1% hoặc trên phân vị 99% của tập train được đưa về đúng ngưỡng đó |

In [34]:
outlier_results = pd.DataFrame({
    "giữ nguyên": quick_scores(make_preprocessor(outlier="none")),
    "log1p": quick_scores(make_preprocessor(outlier="log")),
    f"cắt phân vị {CLIP_QUANTILES[0]:.0%}–{CLIP_QUANTILES[1]:.0%}": quick_scores(make_preprocessor(outlier="clip")),
}).T.drop(columns="số dòng train")
best_outlier = outlier_results["PR-AUC"].idxmax()

# Trước / sau trên train: giá trị lớn nhất của các cột đuôi dài
clipper = QuantileClipper(*CLIP_QUANTILES).fit(X_train[OUTLIER_COLS])
before_after = pd.DataFrame({"max trước": X_train[OUTLIER_COLS].max(), "ngưỡng trên (phân vị 99%)": clipper.upper_,
                             "max sau log1p": np.log1p(X_train[OUTLIER_COLS]).max(),
                             "% dòng train bị cắt": (X_train[OUTLIER_COLS] > clipper.upper_).mean() * 100})

plot_df = outlier_results.reset_index(names="strategy").melt(id_vars="strategy", var_name="metric", value_name="score")
plot_df = plot_df.replace(LABEL_EN)
fig = px.bar(plot_df, x="metric", y="score", color="strategy", barmode="group",
             text=[f"{v:.3f}" for v in plot_df["score"]], color_discrete_sequence=QUALITATIVE)
fig.update_traces(textposition="outside")
show_fig(fig, f"Outlier handling for logistic regression: best PR-AUC with '{best_outlier}'",
         xaxis_title="Metric on validation (cancelled class)", yaxis_title="Score", yaxis_range=[0, 1.05],
         height=460, legend_title_text="")
display(outlier_results.round(4))
before_after.round(2)

,F1 (hủy),recall (hủy),PR-AUC,ROC-AUC
giữ nguyên,0.648,0.805,0.654,0.849
log1p,0.667,0.822,0.674,0.862
cắt phân vị 1%–99%,0.648,0.804,0.654,0.849


,max trước,ngưỡng trên (phân vị 99%),max sau log1p,% dòng train bị cắt
lead_time,737.000,341.000,6.600,1.000
adr,451.500,261.000,6.110,1.000
stays_in_weekend_nights,19.000,4.000,3.000,0.340
stays_in_week_nights,50.000,10.000,3.930,0.410
total_nights,69.000,14.000,4.250,0.440
days_in_waiting_list,391.000,0.000,5.970,0.830


**Nhận xét.** `log1p` cho kết quả tốt nhất với Logistic Regression: PR-AUC tăng từ 0,654 lên 0,674, F1 lớp hủy từ 0,647 lên 0,667 và ROC-AUC từ 0,849 lên 0,862. Các cột như `lead_time` (tới 737 ngày) hay `stays_in_week_nights` (tới 50 đêm) có một số giá trị rất lớn làm ảnh hưởng đến mô hình tuyến tính. Sau khi lấy `log1p`, giá trị lớn nhất của `lead_time` chỉ còn 6,6 và của `adr` còn 6,1.

Cách cắt ngưỡng hầu như không thay đổi kết quả (PR-AUC 0,654) vì chỉ khoảng 0,3–1% số dòng train bị cắt. Riêng `days_in_waiting_list` có phân vị 99% bằng 0, nên nếu cắt thì 0,83% số booking có thời gian chờ sẽ bị đưa hết về 0.

Vì vậy bộ tiền xử lý cuối cùng (`PREP_CHOICE`) dùng median cho cột số, coi missing là một giá trị riêng và lấy `log1p` cho các cột có đuôi dài. Với mô hình dạng cây, `log1p` không làm thay đổi kết quả vì cây chỉ dựa vào thứ tự giá trị, nên lựa chọn này chủ yếu giúp các mô hình tuyến tính.

In [35]:
# Lựa chọn cuối cho mục 7 (giải thích ở nhận xét 6.4 và 6.5)
PREP_CHOICE = dict(num_impute="median",    # 6.4: KNN không khác median (chỉ 4 ô số bị thiếu)
                   cat_impute="constant",  # 6.4: missing = category, giữ được thông tin "không có đại lý / công ty"
                   outlier="log")          # 6.5: log1p tăng PR-AUC 0,654 -> 0,674 với Logistic Regression
preprocessor = make_preprocessor(**PREP_CHOICE)
prep_check = clone(preprocessor).fit(X_train)
X_train_ml = prep_check.transform(X_train)
feature_names = prep_check.get_feature_names_out()
print("Bộ tiền xử lý cho mục 7:", PREP_CHOICE, f"-> {len(feature_names)} cột")

Bộ tiền xử lý cho mục 7: {'num_impute': 'median', 'cat_impute': 'constant', 'outlier': 'log'} -> 81 cột


### 6.6 Phân phối trước và sau tiền xử lý

Hình dưới so sánh phân phối của hai cột số có độ phân tán lớn nhất trước và sau khi lấy `log1p` và chuẩn hóa.

In [36]:
show_cols = X_train[NUM_COLS].std().sort_values(ascending=False).index[:2].tolist()
fig = make_subplots(rows=len(show_cols), cols=2, vertical_spacing=0.2,
                    subplot_titles=[t for c in show_cols for t in (f"{c}: before", f"{c}: after preprocessing")])
for r, c in enumerate(show_cols, start=1):
    j = [k for k, n in enumerate(feature_names) if n.split("__", 1)[1] == c][0]  # cột c nằm ở nhóm num hoặc tail
    fig.add_trace(hist_bar(X_train[c], color=BLUE), row=r, col=1)
    fig.add_trace(hist_bar(X_train_ml[:, j], color=ORANGE), row=r, col=2)
fig.update_layout(showlegend=False)
show_fig(fig, f"log1p + RobustScaler compress the long right tail of {show_cols[0]} and {show_cols[1]}",
         height=300 * len(show_cols))

<a id="s7"></a>
## 7. Mô hình hóa

### 7.1 So sánh các mô hình trên tập validation

Mỗi mô hình được đặt trong một `Pipeline` gồm bộ tiền xử lý đã chọn ở 6.5 và mô hình phân loại. Các mô hình được thử gồm:

| Mô hình | Lý do chọn |
|---|---|
| `DummyClassifier` | Mô hình cơ sở (baseline), luôn đoán lớp đa số là "không hủy" |
| `LogisticRegression` | Mô hình tuyến tính, huấn luyện nhanh và dễ giải thích |
| `DecisionTreeClassifier` | Học được quan hệ phi tuyến, giới hạn `max_depth` để tránh overfitting |
| `RandomForestClassifier` | Kết hợp nhiều cây nên ổn định hơn một cây đơn |
| `HistGradientBoostingClassifier` | Xây các cây nối tiếp, cây sau sửa lỗi của cây trước, phù hợp với dữ liệu bảng lớn |
| Các phiên bản `class_weight="balanced"` | Tăng trọng số cho lớp hủy, vốn chỉ chiếm 27,7% dữ liệu |

Lớp dương được chọn là lớp hủy. Metric chính là PR-AUC (average precision), đánh giá việc sắp xếp xác suất hủy mà không phụ thuộc vào ngưỡng và phản ánh lớp thiểu số tốt hơn ROC-AUC. F1 của lớp hủy đánh giá kết quả dự đoán ở ngưỡng mặc định 0,5. Precision, recall, ROC-AUC, F1-macro và accuracy được ghi thêm để tham khảo, lưu ý rằng baseline đã có accuracy 72,3%.

In [37]:
def evaluate(model, X, y):
    """Metric của một mô hình đã fit trên một tập dữ liệu; lớp dương = lớp hủy (POS)."""
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, POS]
    return {"precision (hủy)": precision_score(y == POS, pred == POS, zero_division=0),
            "recall (hủy)": recall_score(y == POS, pred == POS),
            "F1 (hủy)": f1_score(y == POS, pred == POS),
            "PR-AUC": average_precision_score(y == POS, proba),
            "ROC-AUC": roc_auc_score(y == POS, proba),
            "F1-macro": f1_score(y, pred, average="macro"),
            "accuracy": accuracy_score(y, pred)}


def make_model(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])


MODELS = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Logistic Regression (balanced)": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    "Random Forest (balanced)": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                       n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=SEED),
    "HistGradientBoosting (balanced)": HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
}

fitted, rows = {}, []
for name, estimator in MODELS.items():
    model = make_model(estimator).fit(X_train, y_train)
    fitted[name] = model
    rows.append({"model": name, **evaluate(model, X_val, y_val)})
    print(f"xong: {name}")

results = pd.DataFrame(rows).set_index("model").sort_values("PR-AUC", ascending=False)
results.round(3)

xong: Baseline (majority class)
xong: Logistic Regression
xong: Logistic Regression (balanced)
xong: Decision Tree
xong: Random Forest
xong: Random Forest (balanced)
xong: HistGradientBoosting
xong: HistGradientBoosting (balanced)


,precision (hủy),recall (hủy),F1 (hủy),PR-AUC,ROC-AUC,F1-macro,accuracy
model,,,,,,,
HistGradientBoosting,0.733,0.637,0.682,0.779,0.903,0.785,0.835
HistGradientBoosting (balanced),0.602,0.856,0.707,0.775,0.902,0.779,0.803
Random Forest,0.741,0.648,0.692,0.775,0.902,0.792,0.840
Random Forest (balanced),0.748,0.636,0.687,0.774,0.903,0.790,0.840
Decision Tree,0.719,0.576,0.640,0.701,0.879,0.760,0.820
Logistic Regression,0.680,0.532,0.597,0.683,0.861,0.732,0.801
Logistic Regression (balanced),0.562,0.822,0.667,0.674,0.862,0.747,0.773
Baseline (majority class),0.000,0.000,0.000,0.277,0.500,0.420,0.723


In [38]:
SHOW_METRICS = ["recall (hủy)", "F1 (hủy)", "PR-AUC", "ROC-AUC"]
plot_df = results[SHOW_METRICS].reset_index().melt(id_vars="model", var_name="metric", value_name="score")
plot_df = plot_df.replace(LABEL_EN)
fig = px.bar(plot_df, x="score", y="model", color="metric", orientation="h", barmode="group",
             category_orders={"model": results.index.tolist()[::-1]}, color_discrete_sequence=QUALITATIVE)
fig.update_xaxes(range=[0, 1])
show_fig(fig, f"Validation: best PR-AUC = {results['PR-AUC'].iloc[0]:.3f} ({results.index[0]}), "
              f"baseline PR-AUC = {results.loc['Baseline (majority class)', 'PR-AUC']:.3f}",
         height=560, xaxis_title="Score on validation (cancelled class)", yaxis_title="", legend_title_text="")

**Nhận xét.** Tất cả các mô hình đều tốt hơn nhiều so với baseline. Baseline có PR-AUC 0,277 (bằng tỷ lệ hủy) và F1 lớp hủy bằng 0 dù accuracy là 72,3%.

Các mô hình dạng cây cho kết quả tốt hơn rõ rệt so với Logistic Regression. HistGradientBoosting có PR-AUC cao nhất là 0,779, tiếp theo là Random Forest 0,775, hai phiên bản balanced 0,774–0,775 và Decision Tree 0,701, trong khi Logistic Regression chỉ đạt 0,674–0,683. ROC-AUC của các mô hình cây khoảng 0,90, còn Logistic Regression khoảng 0,86. Điều này phù hợp với EDA: quan hệ giữa `lead_time` và tỷ lệ hủy không tuyến tính, khác nhau giữa hai khách sạn (5.6) và các yếu tố có tác động cộng dồn (5.8).

`class_weight="balanced"` gần như không làm thay đổi PR-AUC mà chủ yếu đổi giữa precision và recall. Với HistGradientBoosting, recall tăng từ 0,637 lên 0,856 còn precision giảm từ 0,733 xuống 0,602, F1 lớp hủy tăng từ 0,682 lên 0,707. Với Logistic Regression, recall tăng từ 0,531 lên 0,822. Mục 7.3 so sánh kỹ hơn cách này với SMOTE và điều chỉnh ngưỡng.

Random Forest có accuracy cao nhất (0,840) nhưng PR-AUC thấp hơn HistGradientBoosting. HistGradientBoosting cũng huấn luyện nhanh hơn và cho mô hình nhỏ hơn, nên được dùng ở các mục 7.3–7.4.

### 7.2 Thí nghiệm leakage

Mục 3 đã loại các cột chỉ có sau khi đặt phòng. Thí nghiệm này kiểm tra xem nếu không loại các cột đó thì kết quả sẽ thay đổi thế nào. Mô hình HistGradientBoosting được huấn luyện trên cùng tập train và validation, lần lượt thêm lại các cột sau:

| Trường hợp | Cột thêm vào | Vì sao không được dùng |
|---|---|---|
| Không thêm | | Chỉ dùng thông tin có lúc đặt phòng (`FEATURES`) |
| Thêm thông tin lúc check-in | `assigned_room_type`, `booking_changes` | Loại phòng thực nhận và số lần sửa booking chỉ biết khi khách đã đến hoặc đã hủy |
| Thêm trạng thái cuối | `reservation_status` | Canceled, No-Show, Check-Out chính là kết quả cần dự đoán |

In [39]:
LEAK_COLS = {"assigned_room_type": "cat", "booking_changes": "num", "reservation_status": "cat"}
leak = raw.loc[raw_index, list(LEAK_COLS)].reset_index(drop=True)  # cùng thứ tự dòng với df


def with_leak(X, cols):
    """Ghép thêm các cột leakage (theo chỉ số dòng của df) vào X."""
    return X.join(leak.loc[X.index, cols]) if cols else X


def leak_preprocessor(cols):
    """Bộ tiền xử lý đã chốt cho FEATURES + mã hóa riêng các cột leakage được thêm vào."""
    parts = [("base", make_preprocessor(**PREP_CHOICE), FEATURES)]
    cat = [c for c in cols if LEAK_COLS[c] == "cat"]
    num = [c for c in cols if LEAK_COLS[c] == "num"]
    if cat:
        parts.append(("leak_cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat))
    if num:
        parts.append(("leak_num", "passthrough", num))
    return ColumnTransformer(parts)


SCENARIOS = {
    "no leakage": [],
    "+ assigned_room_type, booking_changes": ["assigned_room_type", "booking_changes"],
    "+ reservation_status": ["reservation_status"],
}
rows = []
for name, cols in SCENARIOS.items():
    model = Pipeline([("prep", leak_preprocessor(cols)),
                      ("model", HistGradientBoostingClassifier(random_state=SEED))]).fit(with_leak(X_train, cols), y_train)
    rows.append({"scenario": name, **evaluate(model, with_leak(X_val, cols), y_val)})
leak_results = pd.DataFrame(rows).set_index("scenario")

plot_df = leak_results[SHOW_METRICS].reset_index().melt(id_vars="scenario", var_name="metric", value_name="score")
plot_df = plot_df.replace(LABEL_EN)
fig = px.bar(plot_df, x="metric", y="score", color="scenario", barmode="group",
             text=[f"{v:.3f}" for v in plot_df["score"]], color_discrete_sequence=[BLUE, YELLOW, RED])
fig.update_traces(textposition="outside")
show_fig(fig, f"Leaked columns inflate PR-AUC from {leak_results['PR-AUC'].iloc[0]:.3f} to "
              f"{leak_results['PR-AUC'].iloc[1]:.3f} (check-in info) and {leak_results['PR-AUC'].iloc[2]:.3f} (final status)",
         xaxis_title="Metric on validation (cancelled class)", yaxis_title="Score", yaxis_range=[0, 1.1],
         height=460, legend_title_text="")
leak_results.round(3)

,precision (hủy),recall (hủy),F1 (hủy),PR-AUC,ROC-AUC,F1-macro,accuracy
scenario,,,,,,,
no leakage,0.733,0.637,0.682,0.779,0.903,0.785,0.835
"+ assigned_room_type, booking_changes",0.745,0.664,0.702,0.793,0.910,0.798,0.844
+ reservation_status,1.000,1.000,1.000,1.000,1.000,1.000,1.000


**Nhận xét.** Khi thêm `reservation_status`, mọi metric đều bằng 1,000. Mô hình không học được gì về hành vi của khách mà chỉ đọc lại kết quả, vì Canceled và No-Show luôn ứng với 1 còn Check-Out luôn ứng với 0. Trường hợp này dễ nhận ra vì kết quả hoàn hảo một cách bất thường.

Trường hợp thêm `assigned_room_type` và `booking_changes` khó nhận ra hơn. PR-AUC chỉ tăng từ 0,779 lên 0,793, F1 lớp hủy từ 0,682 lên 0,702 và accuracy từ 0,835 lên 0,844. Mức tăng nhỏ như vậy rất dễ bị hiểu là mô hình đã được cải thiện. Tuy nhiên hai cột này chỉ có sau khi khách đến hoặc đã hủy, nên tại thời điểm dự đoán chúng chưa tồn tại và phần cải thiện này sẽ mất khi sử dụng thực tế.

Kết quả này cho thấy việc kiểm tra từng cột có sẵn tại lúc đặt phòng hay không ở mục 3 là cần thiết. Các kết quả từ đây trở đi chỉ dùng các cột trong `FEATURES`.

### 7.3 Xử lý mất cân bằng lớp

Sau khi làm sạch chỉ có 27,7% booking bị hủy, nên mô hình thường nghiêng về dự đoán "không hủy". Bốn cách được so sánh trên cùng loại mô hình có PR-AUC tốt hơn ở 7.1 (giữa Random Forest và HistGradientBoosting):

| Cách | Mô tả |
|---|---|
| Không xử lý | Mô hình mặc định với ngưỡng 0,5 |
| `class_weight="balanced"` | Lỗi trên lớp hủy được nhân trọng số khoảng 72,3 / 27,7 |
| SMOTE | Tạo thêm mẫu hủy bằng cách nội suy giữa các mẫu hủy gần nhau. SMOTE nằm trong pipeline của `imblearn` nên chỉ tạo mẫu trên tập train khi huấn luyện, tập validation giữ nguyên |
| Điều chỉnh ngưỡng | `TunedThresholdClassifierCV` chọn ngưỡng xác suất cho F1 lớp hủy cao nhất. Ngưỡng được chọn bằng cross-validation trên tập train, không dùng tập validation |

In [40]:
plain = results.loc[["Random Forest", "HistGradientBoosting"], "PR-AUC"]
FAMILY = plain.idxmax()                                   # họ mô hình tốt hơn ở 7.1 theo PR-AUC
print(f"Họ mô hình dùng ở 7.3–7.4: {FAMILY} (PR-AUC 7.1: " + ", ".join(f"{k} {v:.3f}" for k, v in plain.items()) + ")")


def make_family(class_weight=None, **params):
    """Một estimator của họ FAMILY."""
    if FAMILY == "Random Forest":
        return RandomForestClassifier(n_estimators=300, class_weight=class_weight, n_jobs=-1, random_state=SEED, **params)
    return HistGradientBoostingClassifier(class_weight=class_weight, random_state=SEED, **params)


cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
IMBALANCE = {
    "no handling": make_model(make_family()),
    "class_weight='balanced'": make_model(make_family("balanced")),
    "SMOTE": ImbPipeline([("prep", clone(preprocessor)), ("smote", SMOTE(random_state=SEED)),
                          ("model", make_family())]),
    "tuned threshold": TunedThresholdClassifierCV(make_model(make_family()), scoring="f1", cv=cv3),
}
rows = []
for name, model in IMBALANCE.items():
    model.fit(X_train, y_train)
    fitted[f"{FAMILY} + {name}"] = model
    rows.append({"strategy": name, **evaluate(model, X_val, y_val)})
    print(f"xong: {name}")
imbalance_results = pd.DataFrame(rows).set_index("strategy")
print(f"Ngưỡng chọn bằng CV trên train: {IMBALANCE['tuned threshold'].best_threshold_:.3f}")

IMB_METRICS = ["precision (hủy)", "recall (hủy)", "F1 (hủy)", "PR-AUC"]
plot_df = imbalance_results[IMB_METRICS].reset_index().melt(id_vars="strategy", var_name="metric", value_name="score")
plot_df = plot_df.replace(LABEL_EN)
fig = px.bar(plot_df, x="metric", y="score", color="strategy", barmode="group",
             text=[f"{v:.3f}" for v in plot_df["score"]], color_discrete_sequence=[MUTED, BLUE, ORANGE, AQUA])
fig.update_traces(textposition="outside")
base_recall = imbalance_results.loc["no handling", "recall (hủy)"]
best_recall = imbalance_results["recall (hủy)"].max()
show_fig(fig, f"{FAMILY}: handling imbalance raises recall of cancellations from {base_recall:.3f} "
              f"to {best_recall:.3f} ({imbalance_results['recall (hủy)'].idxmax()})",
         xaxis_title="Metric on validation (cancelled class)", yaxis_title="Score", yaxis_range=[0, 1.1],
         height=460, legend_title_text="")
imbalance_results.round(3)

Họ mô hình dùng ở 7.3–7.4: HistGradientBoosting (PR-AUC 7.1: Random Forest 0.775, HistGradientBoosting 0.779)
xong: no handling
xong: class_weight='balanced'
xong: SMOTE
xong: tuned threshold
Ngưỡng chọn bằng CV trên train: 0.374


,precision (hủy),recall (hủy),F1 (hủy),PR-AUC,ROC-AUC,F1-macro,accuracy
strategy,,,,,,,
no handling,0.733,0.637,0.682,0.779,0.903,0.785,0.835
class_weight='balanced',0.602,0.856,0.707,0.775,0.902,0.779,0.803
SMOTE,0.688,0.712,0.700,0.770,0.900,0.791,0.831
tuned threshold,0.666,0.770,0.715,0.779,0.903,0.797,0.830


**Nhận xét.** Kết quả của HistGradientBoosting trên tập validation:

| Cách | Precision | Recall | F1 (hủy) | PR-AUC |
|---|---|---|---|---|
| Không xử lý | 0,733 | 0,637 | 0,682 | 0,779 |
| `class_weight="balanced"` | 0,602 | 0,856 | 0,707 | 0,775 |
| SMOTE | 0,688 | 0,712 | 0,700 | 0,770 |
| Điều chỉnh ngưỡng (0,374) | 0,666 | 0,770 | 0,715 | 0,779 |

Cả ba cách đều làm tăng recall của lớp hủy. Mô hình gốc bỏ sót 36,3% booking bị hủy, còn khi điều chỉnh ngưỡng chỉ bỏ sót 23,0%. Điều chỉnh ngưỡng cho F1 cao nhất (0,715) và giữ nguyên PR-AUC, vì mô hình không thay đổi mà chỉ đổi ngưỡng phân loại: booking được dự đoán là hủy khi xác suất từ 0,374 trở lên thay vì 0,5.

`class_weight="balanced"` cho recall cao nhất (0,856) nhưng precision giảm còn 0,602, nghĩa là cứ 10 booking bị cảnh báo thì có khoảng 4 booking thực tế không hủy. SMOTE có PR-AUC thấp nhất (0,770), có thể do việc nội suy trên cả các cột one-hot tạo ra những mẫu không có thật, ví dụ một booking nửa thuộc City Hotel nửa thuộc Resort Hotel.

Do đó mô hình cuối sử dụng cách điều chỉnh ngưỡng (`USE_THRESHOLD = True` ở 7.5). Nếu khách sạn xem việc bỏ sót một booking hủy là tốn kém hơn một lần cảnh báo nhầm, có thể chọn ngưỡng thấp hơn nữa.

### 7.4 Tinh chỉnh siêu tham số

`RandomizedSearchCV` thử ngẫu nhiên 10 tổ hợp siêu tham số. Mỗi tổ hợp được đánh giá bằng cross-validation 3-fold trên tập train theo PR-AUC, tập validation và test không được sử dụng. Cách thử ngẫu nhiên được chọn thay cho `GridSearchCV` vì với cùng thời gian chạy, nó thử được nhiều giá trị hơn cho từng tham số.

In [41]:
if FAMILY == "Random Forest":
    PARAM_SPACE = {"model__max_depth": [None, 20, 30], "model__min_samples_leaf": [1, 2, 5],
                   "model__max_features": ["sqrt", 0.3, 0.5]}
else:
    PARAM_SPACE = {"model__learning_rate": [0.05, 0.1, 0.2], "model__max_iter": [200, 400],
                   "model__max_leaf_nodes": [15, 31, 63], "model__min_samples_leaf": [20, 50, 100],
                   "model__l2_regularization": [0.0, 0.1, 1.0]}
N_ITER = 10  # 10 tổ hợp x 3 fold = 30 lần fit; giữ thời gian chạy hợp lý trên Colab (2 CPU)

search = RandomizedSearchCV(make_model(make_family()), PARAM_SPACE, n_iter=N_ITER, scoring="average_precision",
                            cv=cv3, random_state=SEED, n_jobs=-1)
search.fit(X_train, y_train)

tuned_name = f"{FAMILY} (tuned)"
fitted[tuned_name] = search.best_estimator_
results.loc[tuned_name] = pd.Series(evaluate(search.best_estimator_, X_val, y_val))  # dòng mới, khớp theo tên cột
print("Tham số tốt nhất:", search.best_params_, f"| CV PR-AUC = {search.best_score_:.3f}")
print(f"Validation PR-AUC: mặc định {results.loc[FAMILY, 'PR-AUC']:.3f} -> tuned {results.loc[tuned_name, 'PR-AUC']:.3f}")
(pd.DataFrame(search.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
   .sort_values("rank_test_score").head(5))

Tham số tốt nhất: {'model__min_samples_leaf': 100, 'model__max_leaf_nodes': 31, 'model__max_iter': 200, 'model__learning_rate': 0.1, 'model__l2_regularization': 1.0} | CV PR-AUC = 0.774
Validation PR-AUC: mặc định 0.779 -> tuned 0.783


,params,mean_test_score,std_test_score,rank_test_score
2,"{'model__min_samples_leaf': 100, 'model__max_l...",0.774,0.003,1
4,"{'model__min_samples_leaf': 50, 'model__max_le...",0.773,0.004,2
0,"{'model__min_samples_leaf': 100, 'model__max_l...",0.772,0.004,3
7,"{'model__min_samples_leaf': 20, 'model__max_le...",0.772,0.005,4
8,"{'model__min_samples_leaf': 50, 'model__max_le...",0.771,0.003,5


### 7.5 Chọn mô hình cuối cùng

Mô hình cuối là mô hình đã tinh chỉnh ở 7.4, kết hợp với điều chỉnh ngưỡng nếu ở 7.3 cách này làm tăng F1 lớp hủy. Ngưỡng được chọn lại bằng cross-validation trên tập train cho đúng mô hình đã tinh chỉnh. Sau đó mô hình được huấn luyện lại trên train và validation với siêu tham số và ngưỡng giữ nguyên, rồi đánh giá một lần trên tập test ở mục 8.

In [42]:
USE_THRESHOLD = (imbalance_results.loc["tuned threshold", "F1 (hủy)"]
                 > imbalance_results.loc["no handling", "F1 (hủy)"])
best_pipeline = search.best_estimator_                       # đã fit trên train ở 7.4
BEST_NAME = tuned_name + (" + tuned threshold" if USE_THRESHOLD else "")

if USE_THRESHOLD:
    # Chọn ngưỡng cho đúng mô hình đã tinh chỉnh bằng CV 3-fold trên train (không nhìn validation)
    threshold_cv = TunedThresholdClassifierCV(clone(best_pipeline), scoring="f1", cv=cv3).fit(X_train, y_train)
    THRESHOLD = threshold_cv.best_threshold_
    results.loc[BEST_NAME] = pd.Series(evaluate(threshold_cv, X_val, y_val))
    final_model = FixedThresholdClassifier(clone(best_pipeline), threshold=THRESHOLD)
    print(f"Ngưỡng xác suất hủy (CV trên train): {THRESHOLD:.3f}")
else:
    final_model = clone(best_pipeline)

# Fit lại trên train + validation: siêu tham số và ngưỡng giữ nguyên
final_model.fit(pd.concat([X_train, X_val]), np.concatenate([y_train, y_val]))
baseline_model = fitted["Baseline (majority class)"]
print(f"Mô hình cuối: {BEST_NAME}")
summary_rows = ["Baseline (majority class)", "Logistic Regression (balanced)", FAMILY, tuned_name, BEST_NAME]
results.loc[list(dict.fromkeys(summary_rows))].round(3)

Ngưỡng xác suất hủy (CV trên train): 0.364
Mô hình cuối: HistGradientBoosting (tuned) + tuned threshold


,precision (hủy),recall (hủy),F1 (hủy),PR-AUC,ROC-AUC,F1-macro,accuracy
model,,,,,,,
Baseline (majority class),0.000,0.000,0.000,0.277,0.500,0.420,0.723
Logistic Regression (balanced),0.562,0.822,0.667,0.674,0.862,0.747,0.773
HistGradientBoosting,0.733,0.637,0.682,0.779,0.903,0.785,0.835
HistGradientBoosting (tuned),0.733,0.654,0.691,0.783,0.905,0.791,0.838
HistGradientBoosting (tuned) + tuned threshold,0.660,0.782,0.715,0.783,0.905,0.796,0.828


**Nhận xét.** Bộ siêu tham số tốt nhất ở 7.4 gồm `learning_rate` 0,1, `max_iter` 200, `max_leaf_nodes` 31, `min_samples_leaf` 100 và `l2_regularization` 1,0, với PR-AUC trung bình qua cross-validation là 0,774. Trên tập validation, PR-AUC chỉ tăng từ 0,779 lên 0,783 và F1 lớp hủy từ 0,682 lên 0,691. Năm bộ tham số tốt nhất chỉ chênh nhau dưới 0,003, nên cấu hình mặc định của HistGradientBoosting đã gần tốt nhất với dữ liệu này. Bộ tham số được chọn có lá lớn hơn và regularization mạnh hơn, tức là mô hình đơn giản hơn so với mặc định.

Mô hình cuối là HistGradientBoosting đã tinh chỉnh với ngưỡng 0,364. Trên tập validation, mô hình đạt PR-AUC 0,783, F1 lớp hủy 0,715, recall 0,782 và precision 0,660, so với F1 0,667 và PR-AUC 0,674 của Logistic Regression (balanced) và PR-AUC 0,277 của baseline. Accuracy giảm nhẹ từ 0,838 (ngưỡng 0,5) xuống 0,828, nhưng recall tăng thêm 12,8 điểm phần trăm (từ 0,654 lên 0,782). Đây là lựa chọn chấp nhận thêm một số cảnh báo nhầm để phát hiện được nhiều booking hủy hơn.

Mô hình được huấn luyện lại trên 72.398 dòng của train và validation rồi đánh giá trên tập test ở mục 8.

<a id="s8"></a>
## 8. Đánh giá trên tập test

Tập test chưa được dùng trong bất kỳ bước lựa chọn nào ở trên, nên kết quả dưới đây phản ánh hiệu năng của mô hình trên dữ liệu mới.

### 8.1 Kết quả trên tập test

In [43]:
test_scores = pd.DataFrame({
    "Baseline (majority class), test": evaluate(baseline_model, X_test, y_test),
    f"{BEST_NAME}, validation": results.loc[BEST_NAME],             # mô hình fit trên train (mục 7.5)
    f"{BEST_NAME}, test": evaluate(final_model, X_test, y_test),     # mô hình fit trên train + validation
}).T
y_pred = final_model.predict(X_test)
display(test_scores.round(3))
pd.DataFrame(classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)).T.round(3)

,precision (hủy),recall (hủy),F1 (hủy),PR-AUC,ROC-AUC,F1-macro,accuracy
"Baseline (majority class), test",0.000,0.000,0.000,0.277,0.500,0.420,0.723
"HistGradientBoosting (tuned) + tuned threshold, validation",0.660,0.782,0.715,0.783,0.905,0.796,0.828
"HistGradientBoosting (tuned) + tuned threshold, test",0.649,0.775,0.707,0.775,0.900,0.789,0.822


,precision,recall,f1-score,support
0,0.907,0.840,0.872,"9,240.000"
1,0.649,0.775,0.707,"3,537.000"
accuracy,0.822,0.822,0.822,0.822
macro avg,0.778,0.807,0.789,"12,777.000"
weighted avg,0.836,0.822,0.826,"12,777.000"


**Nhận xét.** Kết quả trên tập test gần như bằng trên tập validation: PR-AUC 0,775 (validation 0,783), F1 lớp hủy 0,707 (0,715), recall 0,775 (0,782), precision 0,649 (0,660), ROC-AUC 0,900 (0,905) và accuracy 0,822 (0,828). Các metric chỉ giảm tối đa 0,011, nên mô hình không bị overfit lên tập validation.

So với baseline, PR-AUC tăng từ 0,277 lên 0,775 (gấp khoảng 2,8 lần) và F1 lớp hủy tăng từ 0 lên 0,707, trong khi accuracy chỉ tăng từ 72,3% lên 82,2%. Nếu chỉ nhìn accuracy sẽ đánh giá thấp mức cải thiện của mô hình. Với lớp không hủy, mô hình có precision 0,907 và recall 0,840.

### 8.2 Confusion matrix

Hàng là lớp thực tế, cột là lớp dự đoán. Mỗi ô ghi số booking và tỷ lệ so với tổng số booking của lớp thực tế đó.

In [44]:
LABELS = {"0": "Not cancelled", "1": "Cancelled"}           # tên hiển thị của hai lớp trên hình
names = [LABELS.get(c, c) for c in CLASS_NAMES]
cm = confusion_matrix(y_test, y_pred)
cm_row = cm / cm.sum(axis=1, keepdims=True) * 100
fig = go.Figure(go.Heatmap(z=cm_row, x=names, y=names, colorscale=SEQ_BLUE, zmin=0, zmax=100,
                           text=[[f"{n:,}<br>({p:.1f}%)" for n, p in zip(rn, rp)] for rn, rp in zip(cm, cm_row)],
                           texttemplate="%{text}", colorbar=dict(title="row %"),
                           hovertemplate="true %{y}, predicted %{x}: %{z:.1f}%<extra></extra>"))
show_fig(fig, f"Test: {cm_row[POS, POS]:.1f}% of cancellations caught, {cm_row[1 - POS, POS]:.1f}% of kept bookings "
              f"flagged by mistake", xaxis_title="Predicted class", yaxis_title="True class",
         yaxis_autorange="reversed", height=460)
pd.DataFrame(cm, index=[f"thật: {n}" for n in names], columns=[f"đoán: {n}" for n in names])

,đoán: Not cancelled,đoán: Cancelled
thật: Not cancelled,7759,1481
thật: Cancelled,796,2741


**Nhận xét.** Trong 3.537 booking bị hủy ở tập test, mô hình phát hiện được 2.741 booking (77,5%) và bỏ sót 796 booking (22,5%). Trong 9.240 booking không hủy, có 1.481 booking (16,0%) bị cảnh báo nhầm. Tổng cộng mô hình cảnh báo 4.222 booking, trong đó 64,9% thực sự bị hủy.

Số cảnh báo nhầm tương đối nhiều là do ngưỡng 0,364 được chọn ở 7.3 để ưu tiên phát hiện booking hủy. Với khách sạn, một cảnh báo nhầm thường chỉ tốn một cuộc gọi xác nhận, còn một booking hủy không được phát hiện có thể khiến phòng bị bỏ trống.

### 8.3 Đường ROC và precision–recall

Đường ROC thể hiện quan hệ giữa tỷ lệ dương đúng và dương sai khi thay đổi ngưỡng. Với dữ liệu mất cân bằng, đường precision–recall cho nhiều thông tin hơn về lớp hủy.

In [45]:
if IS_BINARY:
    proba_pos = final_model.predict_proba(X_test)[:, POS]
    fpr, tpr, _ = roc_curve(y_test == POS, proba_pos)
    precision, recall, _ = precision_recall_curve(y_test == POS, proba_pos)
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"ROC (AUC = {roc_auc_score(y_test == POS, proba_pos):.3f})",
        f"Precision–recall (AP = {average_precision_score(y_test == POS, proba_pos):.3f})"))
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", line=dict(color=BLUE, width=2), name="ROC"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=MUTED, dash="dot"), name="random"), row=1, col=1)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", line=dict(color=ORANGE, width=2), name="PR"), row=1, col=2)
    fig.add_hline(y=np.mean(y_test == POS), line_dash="dot", line_color=MUTED, row=1, col=2)
    fig.update_xaxes(title_text="False positive rate", row=1, col=1)
    fig.update_yaxes(title_text="True positive rate", row=1, col=1)
    fig.update_xaxes(title_text="Recall", row=1, col=2)
    fig.update_yaxes(title_text="Precision", row=1, col=2)
    show_fig(fig, f"Threshold-free view of the final model on test (positive class = '{MINORITY}')", height=440)
else:
    print("Target nhiều lớp: xem báo cáo theo từng lớp ở mục 8.1.")

**Nhận xét.** ROC-AUC trên tập test là 0,900, nghĩa là khi chọn ngẫu nhiên một booking hủy và một booking không hủy, khoảng 90% trường hợp mô hình cho booking hủy xác suất cao hơn. PR-AUC đạt 0,775, so với 0,277 của một mô hình đoán ngẫu nhiên (đường chấm ngang bằng tỷ lệ hủy). Mỗi điểm trên đường precision–recall ứng với một ngưỡng, giảm ngưỡng thì recall tăng và precision giảm. Ngưỡng 0,364 của mô hình cuối ứng với recall 0,775 và precision 0,649.

### 8.4 Mức độ quan trọng của các đặc trưng

Permutation importance được tính bằng cách lần lượt xáo trộn từng cột trên tập test và đo mức giảm của PR-AUC. Cột nào khi bị xáo trộn làm PR-AUC giảm nhiều thì càng quan trọng với mô hình. Cách này áp dụng được cho mọi mô hình và tính trên các cột gốc trước khi one-hot.

In [46]:
# n_jobs=1: chạy song song sẽ chép final_model sang từng tiến trình con và làm đầy bộ nhớ chia sẻ /dev/shm
# của Colab ("No space left on device"). max_samples: mỗi lần xáo trộn chỉ tính trên 5.000 dòng test ngẫu nhiên.
perm = permutation_importance(final_model, X_test, y_test, scoring="average_precision", n_repeats=5,
                              max_samples=min(5000, len(X_test)), random_state=SEED, n_jobs=1)
importance = (pd.DataFrame({"mức giảm PR-AUC trung bình": perm.importances_mean, "độ lệch chuẩn": perm.importances_std},
                           index=X_test.columns).sort_values("mức giảm PR-AUC trung bình").tail(15))
fig = go.Figure(go.Bar(x=importance["mức giảm PR-AUC trung bình"], y=importance.index, orientation="h",
                       error_x=dict(type="data", array=importance["độ lệch chuẩn"], color=MUTED), marker_color=BLUE,
                       hovertemplate="%{y}: %{x:.4f}<extra></extra>"))
show_fig(fig, f"Permutation importance on test: shuffling '{importance.index[-1]}' costs "
              f"{importance.iloc[-1, 0]:.3f} PR-AUC, more than any other column",
         xaxis_title="Drop in PR-AUC when the column is shuffled", height=max(380, 30 * len(importance)))
importance.iloc[::-1].round(4)

,mức giảm PR-AUC trung bình,độ lệch chuẩn
country,0.162,0.009
lead_time,0.127,0.005
agent,0.115,0.005
total_of_special_requests,0.108,0.004
market_segment,0.082,0.005
required_car_parking_spaces,0.078,0.005
previous_cancellations,0.060,0.006
customer_type,0.052,0.005
deposit_type,0.042,0.005
adr,0.025,0.002


**Nhận xét.** Kết quả phù hợp với phần EDA ở mục 5. Năm cột quan trọng nhất là `country` (PR-AUC giảm 0,162), `lead_time` (0,127), `agent` (0,115), `total_of_special_requests` (0,108) và `market_segment` (0,082). Các cột này đều đã thể hiện mối liên hệ với việc hủy phòng ở EDA: khách Bồ Đào Nha hủy 56,6% (5.4), câu hỏi 1 về thời gian đặt trước, liên hệ giữa việc thiếu `agent` và tỷ lệ hủy (4.1), câu hỏi 3 về yêu cầu đặc biệt và nhóm Groups hủy 61,1% (5.4). `required_car_parking_spaces` đứng thứ sáu (0,078), phù hợp với việc không có booking nào yêu cầu chỗ đỗ xe bị hủy (5.3).

`deposit_type` chỉ đứng thứ chín (0,042) dù booking Non Refund hủy tới 95%. Nguyên nhân là sau khi bỏ trùng chỉ còn khoảng 1% booking Non Refund (câu hỏi 2), nên xáo trộn cột này chỉ ảnh hưởng đến rất ít dòng. Ngày đến trong tháng và loại phòng gần như không đóng góp, phù hợp với mục 5.4 khi tỷ lệ hủy theo tháng chỉ dao động 31–42%. Cần lưu ý rằng các cột có liên quan với nhau như `agent`, `market_segment` và `distribution_channel` có thể chia nhau mức độ quan trọng, nên giá trị của từng cột có thể thấp hơn ảnh hưởng thực tế.

### 8.5 Phân tích lỗi

Bảng đầu tiên so sánh bốn nhóm booking (hủy và được phát hiện, hủy nhưng bị bỏ sót, không hủy nhưng bị cảnh báo nhầm, không hủy và được dự đoán đúng) theo các đặc trưng quan trọng ở EDA và 8.4. Bảng thứ hai liệt kê các booking bị bỏ sót mà mô hình dự đoán xác suất hủy thấp nhất.

In [47]:
proba = final_model.predict_proba(X_test)
errors = X_test.assign(true=label_encoder.inverse_transform(y_test),
                       predicted=label_encoder.inverse_transform(y_pred),
                       **{"P(hủy)": proba[:, POS]})
outcome = np.select([(y_test == POS) & (y_pred == POS), (y_test == POS) & (y_pred != POS),
                     (y_test != POS) & (y_pred == POS)],
                    ["hủy, đoán đúng", "hủy, bị bỏ sót (FN)", "không hủy, cảnh báo nhầm (FP)"], "không hủy, đoán đúng")
print(f"{(y_pred != y_test).sum():,} dự đoán sai trên {len(X_test):,} dòng test ({(y_pred != y_test).mean():.1%})")

# Mỗi nhóm kết quả khác nhau thế nào trên các đặc trưng mà EDA và 8.4 cho là quan trọng
profile = X_test.assign(outcome=outcome).groupby("outcome").agg(
    **{"số booking": ("hotel", "size"),
       "median lead_time": ("lead_time", "median"),
       "% Bồ Đào Nha": ("country", lambda s: (s == "PRT").mean() * 100),
       "% Non Refund": ("deposit_type", lambda s: (s == "Non Refund").mean() * 100),
       "% có yêu cầu đặc biệt": ("total_of_special_requests", lambda s: (s > 0).mean() * 100),
       "% khách quen": ("is_repeated_guest", lambda s: s.mean() * 100),
       "% City Hotel": ("hotel", lambda s: (s == "City Hotel").mean() * 100),
       "median adr": ("adr", "median")})
display(profile.round(1))
errors[outcome == "hủy, bị bỏ sót (FN)"].sort_values("P(hủy)").head(5)  # bỏ sót mà mô hình tự tin nhất

2,277 dự đoán sai trên 12,777 dòng test (17.8%)


,số booking,median lead_time,% Bồ Đào Nha,% Non Refund,% có yêu cầu đặc biệt,% khách quen,% City Hotel,median adr
outcome,,,,,,,,
"hủy, bị bỏ sót (FN)",796,35.000,37.200,0.000,66.300,2.500,71.700,108.000
"hủy, đoán đúng",2741,94.000,42.800,5.000,32.000,0.800,64.700,109.800
"không hủy, cảnh báo nhầm (FP)",1481,71.000,40.000,0.300,44.400,0.400,68.300,107.100
"không hủy, đoán đúng",7759,28.000,26.100,0.100,56.100,5.900,56.300,91.000


,hotel,lead_time,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_nights,total_guests,has_agent,has_company,true,predicted,P(hủy)
41247,City Hotel,195,August,33,13,2,1,3,0.000,0,BB,DEU,Online TA,TA/TO,0,0,0,A,No Deposit,7,NaN,0,Transient,106.030,0,1,3,3.000,1,0,1,0,0.005
39787,City Hotel,70,June,25,15,0,2,2,0.000,0,HB,NLD,Offline TA/TO,TA/TO,0,0,0,A,No Deposit,6,NaN,0,Transient-Party,109.000,0,0,2,2.000,1,0,1,0,0.005
41630,City Hotel,178,August,35,26,0,2,2,1.000,0,BB,GBR,Offline TA/TO,TA/TO,0,0,0,E,No Deposit,83,NaN,0,Transient,117.600,0,0,2,3.000,1,0,1,0,0.006
48976,City Hotel,22,June,24,13,0,1,2,0.000,0,BB,LUX,Offline TA/TO,TA/TO,0,0,0,A,No Deposit,28,NaN,0,Transient,99.000,0,0,1,2.000,1,0,1,0,0.007
13310,Resort Hotel,0,January,4,26,0,1,1,0.000,0,BB,PRT,Corporate,Corporate,1,0,5,A,No Deposit,NaN,9,0,Transient,35.000,0,0,1,1.000,0,1,1,0,0.008


**Nhận xét.** 796 booking hủy bị bỏ sót có đặc điểm giống các booking ít khả năng hủy: median thời gian đặt trước chỉ 35 ngày (so với 94 ngày ở các booking hủy được phát hiện), 66,3% có yêu cầu đặc biệt (so với 32,0%), 2,5% là khách quen (so với 0,8%) và không có booking Non Refund nào. Những trường hợp sai rõ nhất có xác suất hủy dự đoán chỉ 0,005–0,008, ví dụ một khách quen đi công tác đã đặt 5 lần trước đó và chưa hủy lần nào. Lý do hủy của các booking này, như thay đổi kế hoạch, ốm đau hay thời tiết, không có trong dữ liệu đặt phòng, nên đây là giới hạn của dữ liệu hơn là của mô hình.

1.481 booking bị cảnh báo nhầm lại có đặc điểm giống booking bị hủy: median thời gian đặt trước 71 ngày, 40,0% là khách Bồ Đào Nha (so với 26,1% ở các booking không hủy được dự đoán đúng) và chỉ 44,4% có yêu cầu đặc biệt. Hai nhóm lỗi có giá phòng tương đương (median 107–108 EUR), nên giá phòng không giải thích được các lỗi này.

<a id="s9"></a>
## 9. Dự đoán trên dữ liệu mới

Ba booking được nhập tay ở dạng dữ liệu thô, sau đó đi qua các bước giống như dữ liệu huấn luyện gồm `normalize_values`, `add_features` và pipeline của mô hình cuối. Những cột không được nhập sẽ được xem là thiếu và xử lý giống lúc huấn luyện.

In [48]:
# Ba booking tự viết, ở dạng thô như lúc nhập vào hệ thống (cột không ghi = không có / thiếu).
new_bookings = pd.DataFrame([
    {"hotel": "City Hotel", "lead_time": 210, "arrival_date_month": "October", "arrival_date_week_number": 42,
     "arrival_date_day_of_month": 15, "stays_in_weekend_nights": 1, "stays_in_week_nights": 2, "adults": 2,
     "children": 0, "babies": 0, "meal": "BB", "country": "PRT", "market_segment": "Groups",
     "distribution_channel": "TA/TO", "is_repeated_guest": 0, "previous_cancellations": 0,
     "previous_bookings_not_canceled": 0, "reserved_room_type": "A", "deposit_type": "Non Refund", "agent": 1,
     "days_in_waiting_list": 0, "customer_type": "Transient", "adr": 75.0, "required_car_parking_spaces": 0,
     "total_of_special_requests": 0},
    {"hotel": "Resort Hotel", "lead_time": 10, "arrival_date_month": "July", "arrival_date_week_number": 28,
     "arrival_date_day_of_month": 10, "stays_in_weekend_nights": 2, "stays_in_week_nights": 5, "adults": 2,
     "children": 1, "babies": 0, "meal": "HB", "country": "GBR", "market_segment": "Direct",
     "distribution_channel": "Direct", "is_repeated_guest": 1, "previous_cancellations": 0,
     "previous_bookings_not_canceled": 3, "reserved_room_type": "D", "deposit_type": "No Deposit",
     "days_in_waiting_list": 0, "customer_type": "Transient", "adr": 180.0, "required_car_parking_spaces": 1,
     "total_of_special_requests": 2},
    {"hotel": "City Hotel", "lead_time": 90, "arrival_date_month": "May", "arrival_date_week_number": 20,
     "arrival_date_day_of_month": 14, "stays_in_weekend_nights": 0, "stays_in_week_nights": 3, "adults": 2,
     "children": 0, "babies": 0, "meal": "BB", "country": "FRA", "market_segment": "Online TA",
     "distribution_channel": "TA/TO", "is_repeated_guest": 0, "previous_cancellations": 0,
     "previous_bookings_not_canceled": 0, "reserved_room_type": "A", "deposit_type": "No Deposit", "agent": 9,
     "days_in_waiting_list": 0, "customer_type": "Transient", "adr": 120.0, "required_car_parking_spaces": 0,
     "total_of_special_requests": 0},
], index=["Đoàn, Non Refund, đặt trước 210 ngày", "Khách quen, đặt trực tiếp, 2 yêu cầu", "Online TA, đặt trước 90 ngày"])

raw_input_cols = [c for c in raw.columns if c not in [TARGET, *DROP_COLS]]             # cột thô đầu vào
new_rows = add_features(normalize_values(new_bookings.reindex(columns=raw_input_cols))).reindex(columns=FEATURES)
pred_proba = final_model.predict_proba(new_rows)
pd.DataFrame({"P(hủy)": pred_proba[:, POS],
              "dự đoán (ngưỡng " + (f"{THRESHOLD:.3f}" if USE_THRESHOLD else "0,5") + ")":
                  np.where(final_model.predict(new_rows) == POS, "hủy", "không hủy")}, index=new_rows.index).round(3)

,P(hủy),dự đoán (ngưỡng 0.364)
"Đoàn, Non Refund, đặt trước 210 ngày",0.998,hủy
"Khách quen, đặt trực tiếp, 2 yêu cầu",0.000,không hủy
"Online TA, đặt trước 90 ngày",0.621,hủy


**Nhận xét.** Booking của đoàn khách Bồ Đào Nha, loại Non Refund, đặt trước 210 ngày có xác suất hủy 0,998 và được dự đoán là hủy, đúng với đặc điểm ở câu hỏi 2. Booking của khách quen đặt trực tiếp, có 2 yêu cầu đặc biệt, có chỗ đỗ xe và đặt trước 10 ngày có xác suất hủy gần như bằng 0, phù hợp với câu hỏi 3 và mục 5.3. Booking của khách Pháp đặt qua Online TA trước 90 ngày, không có yêu cầu đặc biệt có xác suất hủy 0,621. Giá trị này cao hơn ngưỡng 0,364 nên được dự đoán là hủy, nhưng vẫn chưa chắc chắn, và với những booking như vậy khách sạn nên liên hệ xác nhận trước.

<a id="s10"></a>
## 10. Tổng kết

### 10.1 Kết quả

| Mô hình | Tập | Precision (hủy) | Recall (hủy) | F1 (hủy) | PR-AUC | ROC-AUC | Accuracy |
|---|---|---|---|---|---|---|---|
| Baseline (luôn đoán không hủy) | test | 0,000 | 0,000 | 0,000 | 0,277 | 0,500 | 0,723 |
| Logistic Regression (balanced) | validation | 0,562 | 0,822 | 0,667 | 0,674 | 0,862 | 0,773 |
| HistGradientBoosting mặc định | validation | 0,733 | 0,637 | 0,682 | 0,779 | 0,903 | 0,835 |
| HistGradientBoosting tinh chỉnh, ngưỡng 0,364 | validation | 0,660 | 0,782 | 0,715 | 0,783 | 0,905 | 0,828 |
| HistGradientBoosting tinh chỉnh, ngưỡng 0,364 | test | 0,649 | 0,775 | 0,707 | 0,775 | 0,900 | 0,822 |

Dữ liệu sau làm sạch còn 85.175 booking (từ 119.390), được chia theo tỷ lệ 70/15/15. Mô hình cuối được huấn luyện trên train và validation, tập test chỉ dùng một lần để đánh giá.

### 10.2 Các kết quả chính

1. Dòng trùng ảnh hưởng lớn đến các con số thống kê (mục 4.2, Figure 9–11). 34.050 dòng trùng, phần lớn là booking đoàn Non Refund bị hủy, làm tỷ lệ hủy tăng từ 27,7% lên 37,0%. Nếu không bỏ trùng, tỷ lệ hủy của nhóm đặt trước 181–365 ngày ở City Hotel là 62,7% thay vì 45,7%.
2. Booking đặt càng sớm thì càng dễ bị hủy (Figure 9). Ở City Hotel, tỷ lệ hủy tăng từ 10,5% với booking đặt trước tối đa 7 ngày lên 45,7% với booking đặt trước 181–365 ngày.
3. Yêu cầu đặc biệt và khách quen giúp giảm tỷ lệ hủy (Figure 11). Khách mới không có yêu cầu nào hủy 34,7%, có từ 3 yêu cầu trở lên hủy 16,9%. Khách quen hủy 7,7% so với 28,5% ở khách mới.
4. Các cột có sau khi đặt phòng làm kết quả cao hơn thực tế (Figure 16). `reservation_status` cho PR-AUC bằng 1,000, còn hai cột có lúc check-in làm PR-AUC tăng từ 0,779 lên 0,793.
5. Điều chỉnh ngưỡng là cách xử lý mất cân bằng hiệu quả nhất (Figure 17). Recall lớp hủy tăng từ 0,637 lên 0,770, F1 cao nhất (0,715) và PR-AUC không bị giảm như khi dùng SMOTE (0,770).
6. Các đặc trưng quan trọng nhất (Figure 20) là quốc gia, thời gian đặt trước, đại lý và số yêu cầu đặc biệt, phù hợp với kết quả EDA.

### 10.3 Hạn chế

- Dữ liệu chỉ gồm hai khách sạn ở Bồ Đào Nha với ngày đến từ 2015 đến 2017, nên kết quả chưa chắc áp dụng được cho khách sạn ở nơi khác hoặc giai đoạn sau COVID.
- Dataset không có mã booking nên không phân biệt được dòng nhập trùng với các phòng của cùng một đoàn. Quyết định bỏ trùng làm tỷ lệ hủy giảm từ 37,0% xuống 27,7%.
- Cột `country` có thể chứa thông tin chỉ được cập nhật lúc check-in. Nếu đúng như vậy, booking bị hủy sẽ thường bị bỏ trống hoặc ghi giá trị mặc định ở cột này. Dữ liệu không đủ để kiểm chứng, nhưng booking thiếu `country` chỉ hủy 13,7% trong khi khách Bồ Đào Nha hủy 56,6%, và đây lại là cột quan trọng nhất ở 8.4.
- Dữ liệu được chia ngẫu nhiên chứ không theo thời gian. Khi sử dụng thực tế, mô hình phải dự đoán các booking trong tương lai nên kết quả trên test có thể cao hơn thực tế.
- Có 22,5% booking hủy không được phát hiện. Các booking này có đặc điểm giống booking ít khả năng hủy và lý do hủy của khách không có trong dữ liệu.

### 10.4 Hướng mở rộng

- Chia dữ liệu theo thời gian, ví dụ huấn luyện trên ngày đến 2015–2016 và kiểm tra trên 2017, để đánh giá đúng hơn khả năng dự đoán booking trong tương lai.
- Thử bỏ cột `country` hoặc chỉ dùng cột này cho các booking đã có thông tin quốc gia lúc đặt, rồi so sánh PR-AUC để kiểm tra khả năng rò rỉ thông tin.
- Chọn ngưỡng dựa trên chi phí thực tế của khách sạn (chi phí phòng trống và chi phí gọi xác nhận) thay vì tối đa F1.
- Dùng SMOTENC, phiên bản SMOTE dành cho dữ liệu có cột categorical, thay cho SMOTE thông thường.